# Projet de fin de module : pipeline de données de la filière cacao ivoirienne

**Data Engineering - Master Data-AI - Promotion 2024-2025**

|                    |                                                       |
|--------------------|-------------------------------------------------------|
| **Étudiant**       | `KOUADIO Amani Yannick Ivan`                          |
| **Sujet**          | Sujet 2 : pipeline agricole, filière cacao ivoirienne |
| **Dépôt GitHub**   | `https://github.com/gigiyoyo-school/pipeline-cacao-ci.git`   |
| **Filière**        | `Master 1 Data Analyse et Intelligence Artificielle`  |
| **Année scolaire** | `2025-2026`                                           |
| **Enseignant**     | GOUAH Tato Serge                                      |

---

## Contexte

Le Conseil du Café-Cacao suit les apports de fèves des planteurs aux coopératives.
Chaque livraison donne lieu à une pesée : tonnage, taux d'humidité, classement qualité,
prix payé. Ces données arrivent aujourd'hui sous forme de fichiers plats hétérogènes,
saisis sur les bascules des coopératives, et ne permettent aucune analyse consolidée.

Ce projet construit l'entrepôt de données de la filière : du fichier brut au tableau de
bord, en passant par le nettoyage, la modélisation dimensionnelle et l'orchestration.

## Architecture

```
CSV de bascule + référentiels
            |
            v
  Extraction et audit  ......  rapport d'audit
            |
            v
   Nettoyage Pandas    ......  4 familles de défauts traitées
            |
            v
   Enrichissement      ......  6 colonnes calculées
            |
            v
  Contrôle qualité     ......  7 règles bloquantes  --> arrêt si échec
            |
            v
 Schéma en étoile      ......  5 dimensions + 1 table de faits (Supabase)
            |
            v
  Requêtes SQL         ......  JOIN, GROUP BY, CTE, fonctions de fenêtre
            |
            v
  Tableau de bord      ......  6 graphiques
```

L'ensemble est orchestré par un DAG Airflow quotidien et conteneurisé avec Docker.

## Comment lire ce notebook

Le code des cellules est **le code réel du projet**, extrait des scripts du dépôt.
Notebook et scripts ne peuvent donc pas diverger.

Le notebook est **autonome** : il génère ses propres données et ne dépend d'aucun
fichier extérieur. Si la variable d'environnement `SUPABASE_URL` est renseignée, les
tables sont chargées dans Supabase et les requêtes y sont exécutées ; sinon, le même
SQL est exécuté en local par DuckDB. Dans les deux cas, `Run all` se termine sans erreur.

> **Outils d'IA utilisés** : Claude (Anthropic), pour la relecture du code, l'aide au
> débogage et la structuration du projet. Le code a été testé, adapté et compris avant d'être publié sur le repository.

## 0. Configuration

Une seule cellule d'imports, et une boîte à outils minimale reprenant les fonctions
partagées du projet : affichage `rich`, normalisation des libellés, sauvegarde des
étapes intermédiaires en Parquet.

In [21]:
# Dans Google Colab, decommenter la ligne suivante :
# !pip install pandas numpy pyarrow matplotlib rich duckdb sqlalchemy psycopg2-binary -q

from __future__ import annotations

import decimal
import json
import os
import re
import time
import unicodedata
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()
pd.set_option("display.max_columns", None)

# Graine fixe : le dataset genere est reproductible a l'identique
GRAINE = 42
rng = np.random.default_rng(GRAINE)

# Dossiers de travail du notebook, crees a cote de celui-ci
BASE_DIR = Path.cwd() / "data_notebook"
DATA_RAW = BASE_DIR / "raw"
DATA_INTERIM = BASE_DIR / "interim"
DATA_OUTPUT = BASE_DIR / "output"
for dossier in (DATA_RAW, DATA_INTERIM, DATA_OUTPUT):
    dossier.mkdir(parents=True, exist_ok=True)

FICHIER_PESEES = DATA_RAW / "pesees_cacao_ci_80k.csv"
FICHIER_PLANTEURS = DATA_RAW / "referentiel_planteurs.csv"
FICHIER_COOPERATIVES = DATA_RAW / "referentiel_cooperatives.csv"

console.print(f"[green]Dossier de travail :[/] {BASE_DIR}")

Dossier de travail : /Users/yannickivan/Studies/Master 1/Data engineering/pipeline-cacao-ci/data_notebook

In [22]:
# ---------------------------------------------------------------------------
# Boite a outils : les memes fonctions que le module etl_common du depot
# ---------------------------------------------------------------------------
def titre(texte, sous_titre=""):
    console.print()
    console.print(Panel.fit(f"[bold cyan]{texte}[/]", subtitle=sous_titre))

def etape(texte):
    console.print(f"[bold blue]>[/] {texte}")

def ok(texte):
    console.print(f"[bold green]OK[/] {texte}")

def alerte(texte):
    console.print(f"[bold yellow]! [/] {texte}")

def echec(texte):
    console.print(f"[bold red]KO[/] {texte}")


def format_valeur(valeur):
    """Met en forme une valeur pour l'affichage dans une table rich."""
    # PostgreSQL renvoie les colonnes NUMERIC sous forme de Decimal
    if isinstance(valeur, decimal.Decimal):
        valeur = float(valeur)
    if valeur is None or (isinstance(valeur, float) and pd.isna(valeur)):
        return "-"
    if isinstance(valeur, float):
        texte = f"{valeur:,.2f}"
        return texte.rstrip("0").rstrip(".") if "." in texte else texte
    if isinstance(valeur, bool):
        return "oui" if valeur else "non"
    if isinstance(valeur, int):
        return f"{valeur:,}" if abs(valeur) >= 10_000 else str(valeur)
    return str(valeur)


def afficher_df(df, titre_table, max_lignes=15):
    """Affiche un DataFrame sous forme de table rich."""
    table = Table(title=titre_table, header_style="bold magenta")
    for colonne in df.columns:
        justify = "right" if pd.api.types.is_numeric_dtype(df[colonne]) else "left"
        table.add_column(str(colonne), justify=justify, overflow="fold")
    for _, ligne in df.head(max_lignes).iterrows():
        table.add_row(*[format_valeur(v) for v in ligne])
    console.print(table)
    if len(df) > max_lignes:
        console.print(f"[dim]... {len(df) - max_lignes} ligne(s) non affichee(s)[/]")


def taille_lisible(chemin):
    octets = Path(chemin).stat().st_size
    return f"{octets / 1024:.1f} Ko" if octets < 1024**2 else f"{octets / 1024**2:.1f} Mo"


def normaliser(texte):
    """
    Forme canonique d'un libelle : sans accent, en minuscules, ponctuation
    remplacee par des espaces. 'San Pedro', ' SAN  PEDRO ' et 'San-Pedro'
    donnent tous 'san pedro'.
    """
    if not isinstance(texte, str):
        return ""
    decompose = unicodedata.normalize("NFKD", texte)
    sans_accent = "".join(c for c in decompose if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", " ", sans_accent.lower()).strip()


def construire_lookup(valeurs):
    """Dictionnaire forme normalisee -> libelle officiel."""
    return {normaliser(valeur): valeur for valeur in valeurs}


def sauver_etape(df, nom):
    chemin = DATA_INTERIM / f"{nom}.parquet"
    df.to_parquet(chemin, engine="pyarrow", compression="snappy", index=False)
    return chemin


def charger_etape(nom):
    return pd.read_parquet(DATA_INTERIM / f"{nom}.parquet", engine="pyarrow")


def get_csv_path():
    return FICHIER_PESEES


def get_engine():
    """
    Moteur SQLAlchemy vers Supabase, ou None si la connexion n'est pas possible.

    La variable d'environnement est verifiee AVANT d'importer sqlalchemy :
    le notebook doit rester executable dans un environnement ou la
    bibliotheque n'est pas installee.
    """
    url = os.getenv("SUPABASE_URL", "").strip()
    if not url:
        return None
    try:
        import sqlalchemy
    except ImportError:
        alerte("sqlalchemy n'est pas installe : la partie Supabase sera ignoree")
        return None
    if url.startswith("postgresql://"):
        url = url.replace("postgresql://", "postgresql+psycopg2://", 1)
    return sqlalchemy.create_engine(url, pool_pre_ping=True)


ok("Boite a outils prete")

OK Boite a outils prete

## 1. Génération du jeu de données

Le sujet fournit un générateur de départ et invite explicitement à l'enrichir. Son audit
a révélé quatre incohérences qui rendaient impossibles trois des analyses demandées.

| Constat mesuré sur le générateur d'origine | Conséquence |
|---|---|
| Dates étalées du 01/01/2023 au 02/04/2041 | analyse de saisonnalité impossible |
| Grade A à 836 FCFA/kg, Hors grade à 840 | l'analyse « prix par qualité » n'a aucun sens |
| 320 couples région-coopérative pour 40 coopératives | hiérarchie région / coopérative fausse |
| Les 5 000 planteurs livrent dans les 8 régions | `dim_planteur` sans attribut stable |

Cause technique : `pd.date_range(periods=80000, freq='2h')` couvre 18 ans, et
`prix_fcfa_kg` était calculé à partir d'un **second tirage de qualités** indépendant de
la colonne `qualite`.

Le générateur corrigé suit l'ordre réel des événements sur le terrain : qui livre, quand,
avec quelle qualité, donc quelle humidité et quel prix.

In [23]:
# ---------------------------------------------------------------------------
# Parametres generaux
# ---------------------------------------------------------------------------
GRAINE = 42                    # graine fixe : le dataset est reproductible a l'identique
N_PESEES = 80_000              # volume total demande par le sujet
N_PLANTEURS = 5_000
COOP_PAR_REGION = 5

DEBUT = pd.Timestamp("2022-10-01")   # debut de la campagne principale 2022-2023
FIN = pd.Timestamp("2024-09-30")     # fin de la campagne intermediaire 2023-2024

# Proportion de lignes affectees par chaque defaut
TAUX_HUMIDITE_MANQUANTE = 0.04
TAUX_PRIX_ABERRANT = 0.02
TAUX_DOUBLONS = 0.015
TAUX_ERREURS_SAISIE = 0.03

# ---------------------------------------------------------------------------
# Referentiel des regions
# poids : part approximative de la region dans la production nationale.
# Le cacao ivoirien se concentre au Sud-Ouest ; Bondoukou est surtout
# une zone anacarde, d'ou son faible poids.
# ---------------------------------------------------------------------------
REGIONS = {
    # nom            zone de production   port d'export   poids   superficie (ha)
    "Soubre":       ("Sud-Ouest",         "San Pedro",    0.20,   410_000),
    "San Pedro":    ("Sud-Ouest",         "San Pedro",    0.16,   330_000),
    "Daloa":        ("Centre-Ouest",      "San Pedro",    0.14,   295_000),
    "Divo":         ("Centre-Sud",        "Abidjan",      0.13,   240_000),
    "Gagnoa":       ("Centre-Ouest",      "San Pedro",    0.12,   225_000),
    "Abengourou":   ("Est",               "Abidjan",      0.11,   190_000),
    "Aboisso":      ("Sud-Est",           "Abidjan",      0.09,   150_000),
    "Bondoukou":    ("Nord-Est",          "Abidjan",      0.05,    70_000),
}

# ---------------------------------------------------------------------------
# Referentiel des qualites
# Le bareme de prix est exprime en FCFA par kilo. Le taux d'humidite maximum
# de 8 % est la norme d'exportation appliquee en Cote d'Ivoire.
# ---------------------------------------------------------------------------
QUALITES = {
    # grade         rang  humidite cible  prix min  prix max  exportable
    "Grade A":      (1,   6.6,            900,      1100,     True),
    "Grade B":      (2,   7.6,            750,       900,     True),
    "Grade C":      (3,   8.6,            600,       750,     True),
    "Hors grade":   (4,  10.2,            400,       600,     False),
}

# Probabilites des grades selon la campagne. Le cacao seche pendant
# l'harmattan (campagne principale) est de meilleure qualite.
PROBA_QUALITE = {
    "principale":    [0.38, 0.40, 0.18, 0.04],
    "intermediaire": [0.28, 0.38, 0.26, 0.08],
}

# Poids relatif de chaque mois dans les apports annuels.
# Octobre a mars : campagne principale. Avril a septembre : intermediaire.
POIDS_MOIS = {
    10: 0.16, 11: 0.18, 12: 0.15, 1: 0.12, 2: 0.06, 3: 0.04,   # principale
    4: 0.04, 5: 0.06, 6: 0.07, 7: 0.05, 8: 0.03, 9: 0.04,      # intermediaire
}

MOIS_CAMPAGNE_PRINCIPALE = {10, 11, 12, 1, 2, 3}

PRIME_BIO = 1.10               # le cacao certifie bio se paie 10 % de plus
COEF_CAMPAGNE_INTERMEDIAIRE = 0.96   # prix garanti legerement inferieur

# Prime de volume : un gros lot coute moins cher a manipuler et a transporter,
# la cooperative le paie donc legerement mieux. Cet effet cree une correlation
# entre prix et tonnage, sans laquelle la moyenne ponderee par le tonnage
# donnerait exactement le meme resultat que la moyenne simple.
SEUIL_GROS_LOT = 500.0         # kg
SEUIL_PETIT_LOT = 100.0        # kg
PRIME_VOLUME = 1.03
DECOTE_PETIT_LOT = 0.97


# ---------------------------------------------------------------------------
# 1. Referentiel des cooperatives
# ---------------------------------------------------------------------------
def generer_cooperatives(rng: np.random.Generator) -> pd.DataFrame:
    """
    Cree 5 cooperatives par region, soit 40 au total.

    Le code porte le prefixe de sa region : COOP-DAL-003 appartient a Daloa.
    C'est ce lien, absent du generateur d'origine, qui rend la hierarchie
    region / cooperative / planteur exploitable.
    """
    lignes = []
    for region, (zone, port, _, _) in REGIONS.items():
        prefixe = region[:3].upper()
        for numero in range(1, COOP_PAR_REGION + 1):
            lignes.append(
                {
                    "code_cooperative": f"COOP-{prefixe}-{numero:03d}",
                    "nom_cooperative": f"Cooperative {region} {numero}",
                    "region": region,
                    "zone_production": zone,
                    "port_export": port,
                    "annee_creation": int(rng.integers(1995, 2020)),
                    "certifiee_bio": bool(rng.random() < 0.25),
                }
            )
    return pd.DataFrame(lignes)


# ---------------------------------------------------------------------------
# 2. Referentiel des planteurs
# ---------------------------------------------------------------------------
def generer_planteurs(rng: np.random.Generator, cooperatives: pd.DataFrame) -> pd.DataFrame:
    """
    Cree 5 000 planteurs, chacun rattache a une seule cooperative.

    La cooperative est tiree selon le poids de sa region : les regions qui
    produisent le plus comptent le plus de planteurs.

    La certification bio est un attribut du planteur, pas de la livraison :
    une exploitation est certifiee ou elle ne l'est pas. Le generateur
    d'origine la tirait sur chaque ligne de pesee, rendant le meme planteur
    bio un jour et conventionnel le lendemain.
    """
    poids_region = {region: infos[2] for region, infos in REGIONS.items()}
    poids_coop = cooperatives["region"].map(poids_region).to_numpy(dtype=float)
    poids_coop = poids_coop / poids_coop.sum()

    indices = rng.choice(len(cooperatives), size=N_PLANTEURS, p=poids_coop)
    coop_choisies = cooperatives.iloc[indices].reset_index(drop=True)

    # Superficie : loi gamma, moyenne autour de 4 ha, typique des plantations
    # familiales ivoiriennes. On borne a 30 ha pour rester realiste.
    superficie = (rng.gamma(3.0, 1.3, N_PLANTEURS) + 0.5).clip(0.5, 30).round(1)

    # Un planteur d'une cooperative certifiee a plus de chances de l'etre lui-meme
    proba_bio = np.where(coop_choisies["certifiee_bio"], 0.45, 0.06)

    return pd.DataFrame(
        {
            "id_planteur": [f"PLT{i:05d}" for i in range(1, N_PLANTEURS + 1)],
            "code_cooperative": coop_choisies["code_cooperative"],
            "region": coop_choisies["region"],
            "superficie_ha": superficie,
            "annee_adhesion": rng.integers(2005, 2024, N_PLANTEURS),
            "certifie_bio": rng.random(N_PLANTEURS) < proba_bio,
        }
    )


# ---------------------------------------------------------------------------
# 3. Calendrier pondere
# ---------------------------------------------------------------------------
def construire_calendrier(rng: np.random.Generator) -> tuple[pd.DatetimeIndex, np.ndarray]:
    """
    Prepare les jours de la periode et leur probabilite d'apport.

    Deux effets se combinent : la saisonnalite mensuelle (campagne principale
    contre intermediaire) et le rythme hebdomadaire (les cooperatives pesent
    peu le dimanche).
    """
    jours = pd.date_range(DEBUT, FIN, freq="D")

    poids = jours.month.map(POIDS_MOIS).to_numpy(dtype=float)
    poids = poids * np.where(jours.dayofweek == 6, 0.25, 1.0)   # dimanche creux
    poids = poids / poids.sum()

    return jours, poids


# ---------------------------------------------------------------------------
# 4. Les pesees
# ---------------------------------------------------------------------------
def generer_pesees(
    rng: np.random.Generator, planteurs: pd.DataFrame, cooperatives: pd.DataFrame
) -> pd.DataFrame:
    """
    Genere les pesees en suivant l'ordre reel des evenements :
    qui livre, quand, avec quelle qualite, donc quelle humidite et quel prix.
    """
    n_uniques = N_PESEES - int(N_PESEES * TAUX_DOUBLONS)

    # --- Qui livre ---------------------------------------------------------
    # Un planteur avec une grande plantation livre plus souvent.
    poids_planteur = planteurs["superficie_ha"].to_numpy(dtype=float)
    poids_planteur = poids_planteur / poids_planteur.sum()
    indices = rng.choice(len(planteurs), size=n_uniques, p=poids_planteur)
    livreurs = planteurs.iloc[indices].reset_index(drop=True)

    # --- Quand -------------------------------------------------------------
    jours, poids_jours = construire_calendrier(rng)
    dates = jours[rng.choice(len(jours), size=n_uniques, p=poids_jours)]
    est_principale = np.isin(dates.month, list(MOIS_CAMPAGNE_PRINCIPALE))

    # --- Quelle qualite ----------------------------------------------------
    # Deux tirages separes, un par campagne, puis on recombine : c'est ce qui
    # cree la difference de qualite entre les deux saisons.
    grades = np.array(list(QUALITES))
    qualite = np.empty(n_uniques, dtype=object)
    for campagne, masque in (("principale", est_principale), ("intermediaire", ~est_principale)):
        n = int(masque.sum())
        if n:
            qualite[masque] = rng.choice(grades, size=n, p=PROBA_QUALITE[campagne])

    # --- Quelle humidite ---------------------------------------------------
    # L'humidite decoule du grade : c'est elle qui determine le classement
    # sur le terrain, et la correlation doit donc exister dans les donnees.
    cible = np.array([QUALITES[g][1] for g in qualite])
    dispersion = np.where(qualite == "Hors grade", 1.2, 0.5)
    humidite = np.clip(rng.normal(cible, dispersion), 3.5, 16).round(1)

    # --- Quel tonnage ------------------------------------------------------
    # Loi gamma, moyenne autour de 300 kg par apport, modulee par la taille
    # de la plantation. Le tonnage est calcule avant le prix, car il influe
    # sur lui : voir la prime de volume ci-dessous.
    facteur = (livreurs["superficie_ha"].to_numpy() / 4.0) ** 0.6
    tonnage = (rng.gamma(2.0, 130.0, n_uniques) * facteur).clip(5, None).round(1)

    # --- Quel prix ---------------------------------------------------------
    # Prix tire dans les bornes du bareme du grade, puis ajuste par trois
    # effets : la prime bio, le decrochage de la campagne intermediaire, et
    # une prime de volume. Les gros lots coutent moins cher a manipuler et a
    # transporter, la cooperative les paie donc un peu mieux ; les tres petits
    # apports subissent l'effet inverse.
    prix_min = np.array([QUALITES[g][2] for g in qualite])
    prix_max = np.array([QUALITES[g][3] for g in qualite])
    prix = rng.uniform(prix_min, prix_max)
    prix = prix * np.where(livreurs["certifie_bio"].to_numpy(), PRIME_BIO, 1.0)
    prix = prix * np.where(est_principale, 1.0, COEF_CAMPAGNE_INTERMEDIAIRE)
    prix = prix * np.select(
        [tonnage >= SEUIL_GROS_LOT, tonnage < SEUIL_PETIT_LOT],
        [PRIME_VOLUME, DECOTE_PETIT_LOT],
        default=1.0,
    )
    prix = prix.round(0).astype(int)

    pesees = pd.DataFrame(
        {
            "id_pesee": [f"PES{i:07d}" for i in range(1, n_uniques + 1)],
            "date": dates.date,
            "region": livreurs["region"],
            "cooperative": livreurs["code_cooperative"],
            "id_planteur": livreurs["id_planteur"],
            "tonnage_kg": tonnage,
            "qualite": qualite,
            "prix_fcfa_kg": prix,
            "humidite_pct": humidite,
        }
    )

    return pesees.sort_values("date").reset_index(drop=True)


# ---------------------------------------------------------------------------
# 5. Injection des defauts
# ---------------------------------------------------------------------------
def deformer_libelle(texte: str, rng: np.random.Generator) -> str:
    """Reproduit une erreur de saisie plausible sur un nom."""
    variantes = [
        texte.upper(),
        texte.lower(),
        f" {texte}",
        f"{texte} ",
        texte.replace(" ", "  "),
        texte.replace(" ", "-"),
    ]
    return str(rng.choice(variantes))


def injecter_defauts(pesees: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    """
    Ajoute les defauts que le pipeline devra traiter.

    Chaque defaut correspond a une situation reelle de collecte sur bascule :
    capteur d'humidite en panne, prix non saisi, double scan d'un ticket,
    saisie manuelle du nom de la region.
    """
    etape("Injection des defauts de terrain")

    n = len(pesees)

    # --- Humidite manquante : capteur defaillant ou mesure oubliee ---------
    idx = rng.choice(n, size=int(n * TAUX_HUMIDITE_MANQUANTE), replace=False)
    pesees.loc[pesees.index[idx], "humidite_pct"] = np.nan
    console.print(f"  humidite manquante   : {len(idx):,} lignes")

    # --- Prix aberrant : -1 est le code d'erreur du logiciel de bascule ----
    idx = rng.choice(n, size=int(n * TAUX_PRIX_ABERRANT), replace=False)
    pesees.loc[pesees.index[idx], "prix_fcfa_kg"] = -1
    console.print(f"  prix aberrant (-1)   : {len(idx):,} lignes")

    # --- Erreurs de saisie : region et cooperative tapees a la main --------
    idx = rng.choice(n, size=int(n * TAUX_ERREURS_SAISIE), replace=False)
    for position in idx:
        colonne = "region" if rng.random() < 0.6 else "cooperative"
        valeur = pesees.at[pesees.index[position], colonne]
        pesees.at[pesees.index[position], colonne] = deformer_libelle(valeur, rng)
    console.print(f"  erreurs de saisie    : {len(idx):,} lignes")

    # --- Doublons de scan : le meme ticket pese deux fois -------------------
    # Copie a l'identique, identifiant compris : c'est ce qui permet de les
    # detecter par id_pesee.
    n_doublons = N_PESEES - n
    idx = rng.choice(n, size=n_doublons, replace=False)
    doublons = pesees.iloc[idx].copy()
    pesees = pd.concat([pesees, doublons], ignore_index=True)
    console.print(f"  doublons de scan     : {n_doublons:,} lignes")

    return pesees.sort_values("date").reset_index(drop=True)


# ---------------------------------------------------------------------------
# 6. Controles et resume
# ---------------------------------------------------------------------------

In [24]:
# ---------------------------------------------------------------------------
# Execution : referentiels, pesees, defauts, ecriture des trois fichiers
# ---------------------------------------------------------------------------
titre("1. Generation du jeu de donnees", "filiere cacao ivoirienne")

cooperatives = generer_cooperatives(rng)
planteurs = generer_planteurs(rng, cooperatives)
ok(f"{len(cooperatives)} cooperatives et {len(planteurs):,} planteurs")

pesees = generer_pesees(rng, planteurs, cooperatives)
ok(f"{len(pesees):,} pesees du {pesees['date'].min()} au {pesees['date'].max()}")

pesees = injecter_defauts(pesees, rng)

for nom, df_source in {
    "pesees_cacao_ci_80k.csv": pesees,
    "referentiel_planteurs.csv": planteurs,
    "referentiel_cooperatives.csv": cooperatives,
}.items():
    chemin = DATA_RAW / nom
    df_source.to_csv(chemin, index=False, encoding="utf-8")
    ok(f"{nom:<32} {len(df_source):>7,} lignes  {taille_lisible(chemin)}")

╭─────────────────────────────────╮
│ 1. Generation du jeu de donnees │
╰─── filiere cacao ivoirienne ────╯

OK 40 cooperatives et 5,000 planteurs

OK 78,800 pesees du 2022-10-01 au 2024-09-30

> Injection des defauts de terrain

humidite manquante   : 3,152 lignes

...

erreurs de saisie    : 2,364 lignes

doublons de scan     : 1,200 lignes

OK pesees_cacao_ci_80k.csv           80,000 lignes  5.6 Mo

OK referentiel_planteurs.csv          5,000 lignes  218.0 Ko

OK referentiel_cooperatives.csv          40 lignes  2.9 Ko

## 2. Extraction et audit initial

Règle de méthode : cette étape ne modifie rien, elle mesure. Les chiffres produits ici
justifient chaque décision de nettoyage prise ensuite. Un nettoyage dont on ne peut pas
dire ce qu'il a corrigé, et en quelle quantité, n'est pas défendable.

Un libellé mal saisi crée une modalité supplémentaire : comparer le nombre de modalités
brutes au nombre de modalités normalisées révèle les erreurs de saisie sans avoir à les
chercher une par une.

In [25]:
SOURCES = {
    "pesees": FICHIER_PESEES,
    "planteurs": FICHIER_PLANTEURS,
    "cooperatives": FICHIER_COOPERATIVES,
}


def extraire() -> dict[str, pd.DataFrame]:
    """Lit les trois fichiers sources et mesure le cout de l'extraction."""
    etape("Extraction des sources")
    tables = {}

    for nom, chemin in SOURCES.items():
        if not chemin.exists():
            raise FileNotFoundError(
                f"Source absente : {chemin}\nLancez d'abord 01_generer_dataset.py"
            )
        debut = time.time()
        df = pd.read_csv(chemin, encoding="utf-8", low_memory=False)
        duree = time.time() - debut
        memoire = df.memory_usage(deep=True).sum() / 1024**2

        ok(
            f"{nom:<14} {len(df):>7,} lignes x {len(df.columns):>2} colonnes  "
            f"{taille_lisible(chemin):>9}  lu en {duree:.2f}s  ({memoire:.1f} Mo en memoire)"
        )
        tables[nom] = df

    return tables


def auditer_types(df: pd.DataFrame) -> pd.DataFrame:
    """Type detecte par Pandas pour chaque colonne, avec un exemple de valeur."""
    return pd.DataFrame(
        {
            "colonne": df.columns,
            "type_pandas": [str(t) for t in df.dtypes],
            "exemple": [str(df[c].dropna().iloc[0])[:24] if df[c].notna().any() else "-"
                        for c in df.columns],
            "valeurs_distinctes": [df[c].nunique() for c in df.columns],
        }
    )


def auditer_pesees(pesees: pd.DataFrame) -> dict:
    """Audit detaille de la table de faits, sans aucune modification."""
    constats = {}

    # --- Types --------------------------------------------------------------
    etape("Types de donnees")
    afficher_df(auditer_types(pesees), "Structure du fichier des pesees", max_lignes=12)
    alerte("La colonne date est de type texte, elle sera convertie a l'etape suivante.")

    # --- Valeurs manquantes -------------------------------------------------
    etape("Valeurs manquantes")
    manquants = pesees.isna().sum()
    manquants = manquants[manquants > 0]
    if manquants.empty:
        ok("Aucune valeur manquante")
        constats["valeurs_manquantes"] = {}
    else:
        tableau = pd.DataFrame(
            {
                "colonne": manquants.index,
                "nb_manquants": manquants.to_numpy(),
                "part_pct": (manquants / len(pesees) * 100).round(2).to_numpy(),
            }
        )
        afficher_df(tableau, "Valeurs manquantes")
        constats["valeurs_manquantes"] = dict(
            zip(tableau["colonne"], tableau["nb_manquants"].astype(int))
        )

    # --- Doublons -----------------------------------------------------------
    etape("Doublons")
    nb_doublons_id = int(pesees["id_pesee"].duplicated().sum())
    nb_doublons_complets = int(pesees.duplicated().sum())
    console.print(f"  doublons sur id_pesee     : {nb_doublons_id:,}")
    console.print(f"  lignes entierement identiques : {nb_doublons_complets:,}")
    if nb_doublons_id == nb_doublons_complets and nb_doublons_id:
        ok("Tous les doublons sont des copies exactes : double scan du meme ticket")
    constats["doublons_id_pesee"] = nb_doublons_id
    constats["doublons_lignes_completes"] = nb_doublons_complets

    # --- Coherence des libelles ---------------------------------------------
    # Un libelle mal saisi cree une modalite supplementaire : c'est ainsi qu'on
    # detecte les erreurs de saisie sans les chercher une par une.
    etape("Coherence des libelles categoriels")
    lignes = []
    for colonne in ["region", "cooperative", "qualite"]:
        brut = pesees[colonne].nunique()
        normalise = pesees[colonne].map(normaliser).nunique()
        lignes.append(
            {
                "colonne": colonne,
                "modalites_brutes": brut,
                "modalites_normalisees": normalise,
                "variantes_de_saisie": brut - normalise,
            }
        )
    afficher_df(pd.DataFrame(lignes), "Erreurs de saisie detectees")

    exemples = sorted(
        v for v in pesees["region"].unique() if v != v.strip() or v != v.title()
    )[:6]
    console.print(f"  exemples de variantes : {exemples}")
    constats["variantes_saisie"] = {
        ligne["colonne"]: int(ligne["variantes_de_saisie"]) for ligne in lignes
    }

    # --- Statistiques des mesures -------------------------------------------
    etape("Statistiques des mesures")
    stats = pesees[["tonnage_kg", "prix_fcfa_kg", "humidite_pct"]].describe().T
    stats = stats.round(1).reset_index().rename(columns={"index": "mesure"})
    afficher_df(stats, "Distribution des mesures")

    nb_prix_negatifs = int((pesees["prix_fcfa_kg"] <= 0).sum())
    if nb_prix_negatifs:
        alerte(
            f"{nb_prix_negatifs:,} prix negatifs ou nuls : code d'erreur -1 du logiciel "
            "de bascule, a traiter comme une valeur manquante"
        )
    constats["prix_aberrants"] = nb_prix_negatifs

    # --- Periode couverte ---------------------------------------------------
    dates = pd.to_datetime(pesees["date"])
    console.print(
        f"  periode : du [cyan]{dates.min().date()}[/] au [cyan]{dates.max().date()}[/] "
        f"({(dates.max() - dates.min()).days} jours)"
    )
    constats["periode"] = {"debut": str(dates.min().date()), "fin": str(dates.max().date())}

    return constats


def auditer_integrite(tables: dict[str, pd.DataFrame]) -> dict:
    """
    Verifie que les cles des pesees existent bien dans les referentiels.

    On compare sur la forme normalisee : sinon les erreurs de saisie feraient
    passer pour orphelines des lignes parfaitement valides.
    """
    etape("Integrite referentielle")
    pesees, planteurs, cooperatives = tables["pesees"], tables["planteurs"], tables["cooperatives"]

    planteurs_connus = set(planteurs["id_planteur"])
    coops_connues = {normaliser(c) for c in cooperatives["code_cooperative"]}
    regions_connues = {normaliser(r) for r in cooperatives["region"]}

    orphelins_planteur = int((~pesees["id_planteur"].isin(planteurs_connus)).sum())
    orphelins_coop = int((~pesees["cooperative"].map(normaliser).isin(coops_connues)).sum())
    orphelines_region = int((~pesees["region"].map(normaliser).isin(regions_connues)).sum())

    lignes = [
        {"cle": "id_planteur", "orphelins": orphelins_planteur, "reference": "referentiel_planteurs"},
        {"cle": "cooperative", "orphelins": orphelins_coop, "reference": "referentiel_cooperatives"},
        {"cle": "region", "orphelins": orphelines_region, "reference": "referentiel_cooperatives"},
    ]
    afficher_df(pd.DataFrame(lignes), "Cles orphelines apres normalisation")

    if orphelins_planteur or orphelins_coop or orphelines_region:
        alerte("Des cles ne trouvent pas leur correspondance, a investiguer avant chargement")
    else:
        ok("Toutes les cles des pesees existent dans les referentiels")

    return {
        "orphelins_planteur": orphelins_planteur,
        "orphelins_cooperative": orphelins_coop,
        "orphelins_region": orphelines_region,
    }

In [26]:
# ---------------------------------------------------------------------------
# Execution de l'audit
# ---------------------------------------------------------------------------
titre("2. Extraction et audit initial", "aucune donnee n'est modifiee ici")

tables = extraire()
constats = auditer_pesees(tables["pesees"])
constats["integrite"] = auditer_integrite(tables)

for nom, df_source in tables.items():
    sauver_etape(df_source, f"brut_{nom}")

╭────────────────────────────────╮
│ 2. Extraction et audit initial │
╰─ aucune donnee n'est modifiee ─╯

> Extraction des sources

OK pesees          80,000 lignes x  9 colonnes     5.6 Mo  lu en 0.05s  (9.6 Mo en memoire)

OK planteurs        5,000 lignes x  6 colonnes   218.0 Ko  lu en 0.00s  (0.3 Mo en memoire)

OK cooperatives        40 lignes x  7 colonnes     2.9 Ko  lu en 0.00s  (0.0 Mo en memoire)

...

!  1,609 prix negatifs ou nuls : code d'erreur -1 du logiciel de bascule, a traiter comme une valeur manquante

periode : du 2022-10-01 au 2024-09-30 (730 jours)

> Integrite referentielle

         Cles orphelines apres normalisation          
┏━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ cle         ┃ orphelins ┃ reference                ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ id_planteur │         0 │ referentiel_planteurs    │
│ cooperative │         0 │ referentiel_cooperatives │
│ region      │         0 │ referentiel_cooperatives │
└─────────────┴───────────┴──────────────────────────┘

OK Toutes les cles des pesees existent dans les referentiels

## 3. Nettoyage et enrichissement

Quatre familles de défauts, traitées dans un ordre qui n'est pas arbitraire :

1. **normalisation des libellés** avant tout, sinon les jointures avec les référentiels
   échouent sur les variantes de saisie ;
2. **dédoublonnage** avant les imputations, pour ne pas calculer des médianes sur des
   lignes comptées deux fois ;
3. **conversion des types** avant le calcul des campagnes ;
4. **imputation** en dernier, une fois le périmètre stabilisé.

Décision structurante : on impute et on marque, **on ne supprime pas**. Une pesée dont le
prix n'a pas été saisi reste une pesée dont le tonnage est valide ; la supprimer ferait
disparaître 2 % du tonnage réel des analyses de production.

Limite assumée : l'humidité détermine le grade sur le terrain, donc l'imputer par la
médiane du grade est circulaire. Les colonnes `prix_impute` et `humidite_imputee`
permettent d'écarter ces lignes de toute analyse d'un simple filtre.

Six colonnes calculées sont ajoutées, là où le barème en demande quatre. La plus
importante est `montant_fcfa` : sans elle, la moyenne pondérée par le tonnage est
impossible à calculer.

In [27]:
# Bornes des categories de tonnage, exprimees en kilogrammes
BINS_TONNAGE = [0, 100, 500, 1_000, float("inf")]
LABELS_TONNAGE = ["Petit (<100 kg)", "Moyen (100-500 kg)",
                  "Gros (500-1000 kg)", "Tres gros (>1000 kg)"]

# La campagne principale ivoirienne court d'octobre a mars
MOIS_CAMPAGNE_PRINCIPALE = {10, 11, 12, 1, 2, 3}

HUMIDITE_NORME_EXPORT = 8.0


# ---------------------------------------------------------------------------
# 1. Normalisation des libelles
# ---------------------------------------------------------------------------
def normaliser_libelles(
    pesees: pd.DataFrame, cooperatives: pd.DataFrame
) -> tuple[pd.DataFrame, dict]:
    """
    Ramene chaque libelle saisi a sa forme officielle.

    La methode ne code aucune variante en dur. On reduit le libelle saisi et le
    libelle officiel a la meme forme canonique (sans accent, minuscules, sans
    ponctuation), puis on remplace par l'officiel. Une nouvelle faute de frappe
    jamais rencontree sera rattrapee sans modifier le code.
    """
    etape("Normalisation des libelles")

    lookup_region = construire_lookup(cooperatives["region"].unique())
    lookup_coop = construire_lookup(cooperatives["code_cooperative"].unique())

    stats = {}
    for colonne, lookup in (("region", lookup_region), ("cooperative", lookup_coop)):
        avant = pesees[colonne].nunique()
        corrigees = pesees[colonne].map(normaliser).map(lookup)

        # Un libelle sans correspondance reste tel quel : il doit rester
        # visible, la regle R4 le signalera plutot que de le masquer.
        non_resolus = int(corrigees.isna().sum())
        pesees[colonne] = corrigees.fillna(pesees[colonne])

        apres = pesees[colonne].nunique()
        stats[colonne] = {"modalites_avant": int(avant), "modalites_apres": int(apres),
                          "non_resolus": non_resolus}
        console.print(
            f"  {colonne:<12} : {avant:>4} modalites -> {apres:>3} "
            f"({avant - apres} variantes corrigees, {non_resolus} non resolues)"
        )

    return pesees, stats


# ---------------------------------------------------------------------------
# 2. Dedoublonnage
# ---------------------------------------------------------------------------
def dedoublonner(pesees: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Supprime les doublons de scan en conservant la premiere occurrence.

    Le dedoublonnage passe AVANT les imputations : sinon les medianes seraient
    calculees sur des lignes comptees deux fois, ce qui les biaiserait
    legerement, et surtout le volume total serait fausse.
    """
    etape("Dedoublonnage")

    avant = len(pesees)
    exacts = int(pesees.duplicated().sum())
    pesees = pesees.drop_duplicates(subset="id_pesee", keep="first").copy()
    supprimes = avant - len(pesees)

    console.print(f"  lignes entierement identiques : {exacts:,}")
    console.print(f"  lignes supprimees            : {supprimes:,} ({supprimes / avant * 100:.2f} %)")
    ok(f"{len(pesees):,} pesees uniques conservees")

    return pesees, {"lignes_avant": avant, "doublons_supprimes": supprimes,
                    "lignes_apres": len(pesees)}


# ---------------------------------------------------------------------------
# 3. Types et colonnes temporelles
# ---------------------------------------------------------------------------
def corriger_types(pesees: pd.DataFrame) -> pd.DataFrame:
    """
    Convertit la date et derive les colonnes temporelles.

    Le format est impose explicitement plutot que devine : sur un fichier au
    format americain, Pandas peut inverser jour et mois pour toutes les dates
    anterieures au 13 du mois, sans le signaler.
    """
    etape("Conversion des types")

    pesees["date"] = pd.to_datetime(pesees["date"], format="%Y-%m-%d")

    pesees["annee"] = pesees["date"].dt.year
    pesees["mois"] = pesees["date"].dt.month
    pesees["jour_semaine"] = pesees["date"].dt.day_name()

    ok(f"date convertie en {pesees['date'].dtype}")
    return pesees


# ---------------------------------------------------------------------------
# 4. Enrichissement
# ---------------------------------------------------------------------------
def enrichir(pesees: pd.DataFrame, planteurs: pd.DataFrame,
             cooperatives: pd.DataFrame) -> pd.DataFrame:
    """
    Ajoute les attributs des referentiels et les six colonnes calculees.

    Les campagnes sont calculees avant l'imputation des prix : la mediane
    servant a imputer est celle du meme grade sur la meme campagne, ce qui est
    plus juste qu'une mediane globale puisque le prix garanti varie d'une
    campagne a l'autre.
    """
    etape("Enrichissement")

    # --- Jointure avec les referentiels ------------------------------------
    # Le referentiel fait foi : la certification bio et la superficie sont des
    # proprietes de l'exploitation, elles n'ont pas leur place sur la ligne de
    # pesee mais sur le planteur.
    pesees = pesees.merge(
        planteurs[["id_planteur", "superficie_ha", "annee_adhesion", "certifie_bio"]],
        on="id_planteur",
        how="left",
        validate="many_to_one",   # garantit qu'un planteur n'apparait qu'une fois au referentiel
    )
    pesees = pesees.merge(
        cooperatives[["code_cooperative", "zone_production", "port_export"]],
        left_on="cooperative",
        right_on="code_cooperative",
        how="left",
        validate="many_to_one",
    ).drop(columns="code_cooperative")

    # --- Colonne 1 et 2 : campagne et saison -------------------------------
    # Une saison de campagne court d'octobre a septembre : une pesee de
    # janvier 2023 appartient a la saison 2022-2023, pas a 2023-2024.
    pesees["campagne"] = np.where(
        pesees["mois"].isin(MOIS_CAMPAGNE_PRINCIPALE), "Principale", "Intermediaire"
    )
    annee_saison = np.where(pesees["mois"] >= 10, pesees["annee"], pesees["annee"] - 1)
    pesees["saison"] = [f"{a}-{a + 1}" for a in annee_saison]

    # --- Colonne 3 : conformite a la norme d'exportation -------------------
    pesees["conforme_export"] = pesees["humidite_pct"] <= HUMIDITE_NORME_EXPORT

    # --- Colonne 4 : categorie de tonnage ----------------------------------
    pesees["categorie_tonnage"] = pd.cut(
        pesees["tonnage_kg"], bins=BINS_TONNAGE, labels=LABELS_TONNAGE, right=True
    ).astype(str)

    ok("attributs des referentiels et colonnes temporelles ajoutes")
    return pesees


def imputer(pesees: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Impute les valeurs manquantes et marque chaque ligne concernee.

    Le drapeau est ce qui rend l'imputation defendable : toute analyse peut
    ecarter les valeurs imputees d'un simple filtre, et le rapport peut dire
    exactement combien de lignes ne sont pas des mesures reelles.
    """
    etape("Imputation des valeurs manquantes")
    stats = {}

    # --- Prix : -1 est un code d'erreur, pas un prix -----------------------
    pesees["prix_impute"] = pesees["prix_fcfa_kg"] <= 0
    pesees.loc[pesees["prix_impute"], "prix_fcfa_kg"] = np.nan

    mediane_prix = pesees.groupby(["qualite", "campagne"])["prix_fcfa_kg"].transform("median")
    pesees["prix_fcfa_kg"] = pesees["prix_fcfa_kg"].fillna(mediane_prix).round(0).astype(int)

    nb_prix = int(pesees["prix_impute"].sum())
    console.print(
        f"  prix     : {nb_prix:,} valeur(s) imputee(s) par la mediane du grade et de la campagne"
    )
    stats["prix_impute"] = nb_prix

    # --- Humidite : capteur defaillant ou mesure oubliee -------------------
    pesees["humidite_imputee"] = pesees["humidite_pct"].isna()
    mediane_humidite = pesees.groupby("qualite")["humidite_pct"].transform("median")
    pesees["humidite_pct"] = pesees["humidite_pct"].fillna(mediane_humidite).round(1)

    nb_humidite = int(pesees["humidite_imputee"].sum())
    console.print(f"  humidite : {nb_humidite:,} valeur(s) imputee(s) par la mediane du grade")
    stats["humidite_imputee"] = nb_humidite

    # La conformite export doit etre recalculee : elle depend de l'humidite
    pesees["conforme_export"] = pesees["humidite_pct"] <= HUMIDITE_NORME_EXPORT

    alerte(
        "L'humidite determine le grade : l'imputer par la mediane du grade est "
        "circulaire. Les analyses portant sur l'humidite ecarteront ces lignes."
    )
    return pesees, stats


def calculer_mesures(pesees: pd.DataFrame) -> pd.DataFrame:
    """
    Calcule les deux dernieres colonnes, apres imputation puisqu'elles
    dependent du prix.
    """
    etape("Calcul des mesures derivees")

    # --- Colonne 5 : montant de la transaction -----------------------------
    # C'est la colonne qui rend possible la moyenne ponderee. Le prix moyen par
    # qualite ne se calcule pas en AVG(prix) : une pesee de 12 kg pese alors
    # autant qu'une pesee de 2 000 kg. Le prix moyen reel est
    # SUM(montant) / SUM(tonnage), ce qui exige de stocker le montant.
    pesees["montant_fcfa"] = (pesees["tonnage_kg"] * pesees["prix_fcfa_kg"]).round(0).astype(int)

    # --- Colonne 6 : ecart au prix median du grade -------------------------
    mediane_grade = pesees.groupby("qualite")["prix_fcfa_kg"].transform("median")
    pesees["ecart_prix_grade_pct"] = (
        (pesees["prix_fcfa_kg"] - mediane_grade) / mediane_grade * 100
    ).round(1)

    ok("montant_fcfa et ecart_prix_grade_pct calcules")
    return pesees


# ---------------------------------------------------------------------------
# 5. Controle qualite et rapport
# ---------------------------------------------------------------------------

In [28]:
# ---------------------------------------------------------------------------
# Execution du nettoyage et de l'enrichissement
# ---------------------------------------------------------------------------
titre("3. Nettoyage et enrichissement")

pesees = charger_etape("brut_pesees")
planteurs = charger_etape("brut_planteurs")
cooperatives = charger_etape("brut_cooperatives")
nb_depart = len(pesees)

pesees, stats_libelles = normaliser_libelles(pesees, cooperatives)
pesees, stats_doublons = dedoublonner(pesees)
pesees = corriger_types(pesees)
pesees = enrichir(pesees, planteurs, cooperatives)
pesees, stats_imputation = imputer(pesees)
pesees = calculer_mesures(pesees)

sauver_etape(pesees, "pesees_propres")
ok(f"{len(pesees):,} lignes x {len(pesees.columns)} colonnes (depart : {nb_depart:,} x 9)")

afficher_df(
    pesees[["id_pesee", "date", "region", "qualite", "tonnage_kg",
            "prix_fcfa_kg", "montant_fcfa", "campagne", "conforme_export"]].head(5),
    "Cinq premieres lignes du jeu propre",
)

╭────────────────────────────────╮
│ 3. Nettoyage et enrichissement │
╰────────────────────────────────╯

> Normalisation des libelles

region       :   42 modalites ->   8 (34 variantes corrigees, 0 non resolues)

cooperative  :  151 modalites ->  40 (111 variantes corrigees, 0 non resolues)

> Dedoublonnage

...

!  L'humidite determine le grade : l'imputer par la mediane du grade est circulaire. Les analyses portant sur 
l'humidite ecarteront ces lignes.

> Calcul des mesures derivees

OK montant_fcfa et ecart_prix_grade_pct calcules

OK 78,800 lignes x 25 colonnes (depart : 80,000 x 9)

                                        Cinq premieres lignes du jeu propre                                        
┏━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃            ┃            ┃           ┃         ┃            ┃ prix_fcfa_ ┃ montant_fc ┃            ┃ conforme_ex ┃
┃ id_pesee   ┃ date       ┃ region    ┃ qualite ┃ tonnage_kg ┃         kg ┃         fa ┃ campagne   ┃        port ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ PES0048162 │ 2022-10-01 │ Aboisso   │ Grade B │    1,251.3 │        821 │  1,027,317 │ Principale │         oui │
│            │ 00:00:00   │           │         │            │            │            │            │             │
│ PES0025497 │ 2022-10-01 │ Soubre    │ Grade A │      642.3 │       1122 │    720,661 │ Principale │         oui │
│            │ 00:00:00   │           │         │            │            │            │            │             │
│ PES0056753 │ 2022-10-01 │ Soubre    │ Grade B │      299.7 │        890 │    266,733 │ Principale │         oui │
│            │ 00:00:00   │           │         │            │            │            │            │             │
│ PES0003782 │ 2022-10-01 │ Daloa     │ Grade B │      377.4 │        814 │    307,204 │ Principale │         non │
│            │ 00:00:00   │           │         │            │            │            │            │             │
│ PES0021402 │ 2022-10-01 │ Bondoukou │ Grade C │      266.9 │        768 │    204,979 │ Principale │         non │
│            │ 00:00:00   │           │         │            │            │            │            │             │
└────────────┴────────────┴───────────┴─────────┴────────────┴────────────┴────────────┴────────────┴─────────────┘

## 4. Contrôle qualité et détection d'anomalies

Sept règles de validation, là où le barème en demande cinq. La fonction ne lève jamais
d'exception : elle renvoie la liste complète des problèmes, et c'est l'appelant, ici le
DAG Airflow, qui décide d'arrêter le pipeline. Ce partage des rôles permet de
journaliser tous les défauts d'un coup au lieu de s'arrêter au premier.

`R7` est le garde-fou du schéma en étoile : la table de faits portera à la fois la région
et la coopérative, rien n'interdit techniquement qu'elles se contredisent.

La détection d'anomalies est séparée des règles. **Détecter n'est pas supprimer** : un
apport de 2 400 kg est statistiquement rare mais parfaitement possible pour une grande
plantation.

In [29]:
# ---------------------------------------------------------------------------
# Seuils. Ils sont passables en argument pour pouvoir etre durcis en
# production sans toucher au code des regles.
# ---------------------------------------------------------------------------
SEUIL_LIGNES_MINIMUM = 70_000
COLONNES_CRITIQUES = ["id_pesee", "date", "region", "cooperative", "id_planteur",
                      "tonnage_kg", "qualite", "prix_fcfa_kg"]

GRADES_ATTENDUS = {"Grade A", "Grade B", "Grade C", "Hors grade"}

BORNES = {
    # colonne          minimum  maximum   commentaire
    "tonnage_kg":      (1.0,    5_000.0),   # un apport plausible sur une bascule
    "prix_fcfa_kg":    (200.0,  2_000.0),   # bornes larges autour du bareme CCC
    "humidite_pct":    (0.0,    25.0),      # au-dela, la mesure est invalide
}

# Norme d'exportation ivoirienne : au-dela de 8 %, le lot est refuse a l'export
HUMIDITE_NORME_EXPORT = 8.0


@dataclass
class ResultatQualite:
    """Resultat structure d'un controle qualite."""

    controles: list[str] = field(default_factory=list)
    erreurs: list[str] = field(default_factory=list)
    avertissements: list[str] = field(default_factory=list)
    nb_lignes: int = 0

    @property
    def est_valide(self) -> bool:
        """Vrai si aucune regle bloquante n'a echoue."""
        return not self.erreurs

    def resume(self) -> str:
        etat = "QUALITE OK" if self.est_valide else "QUALITE KO"
        return (
            f"{etat} : {len(self.controles)} regle(s) passee(s), "
            f"{len(self.erreurs)} erreur(s), {len(self.avertissements)} avertissement(s) "
            f"sur {self.nb_lignes:,} lignes"
        )

    def en_dict(self) -> dict:
        """Forme serialisable, pour le rapport JSON."""
        return {
            "valide": self.est_valide,
            "nb_lignes": self.nb_lignes,
            "controles_passes": self.controles,
            "erreurs": self.erreurs,
            "avertissements": self.avertissements,
        }


def valider(
    df: pd.DataFrame,
    planteurs: pd.DataFrame | None = None,
    cooperatives: pd.DataFrame | None = None,
    seuil_lignes: int = SEUIL_LIGNES_MINIMUM,
) -> ResultatQualite:
    """
    Applique les sept regles de validation.

    La fonction ne leve jamais d'exception : elle renvoie la liste complete des
    problemes. C'est a l'appelant de decider d'arreter le pipeline. Cela permet
    de voir tous les defauts d'un coup au lieu de s'arreter au premier, et rend
    la fonction testable sans capture d'exception.
    """
    resultat = ResultatQualite(nb_lignes=len(df))

    # --- R1 : volume minimal -----------------------------------------------
    if len(df) < seuil_lignes:
        resultat.erreurs.append(
            f"R1 volume : {len(df):,} lignes, en dessous du seuil de {seuil_lignes:,}. "
            "Extraction probablement incomplete."
        )
    else:
        resultat.controles.append(f"R1 volume : {len(df):,} lignes (seuil {seuil_lignes:,})")

    # --- R2 : unicite de la cle primaire -----------------------------------
    nb_doublons = int(df["id_pesee"].duplicated().sum())
    if nb_doublons:
        resultat.erreurs.append(
            f"R2 unicite : {nb_doublons:,} doublon(s) sur id_pesee. "
            "Le dedoublonnage n'a pas ete applique."
        )
    else:
        resultat.controles.append("R2 unicite : aucun doublon sur id_pesee")

    # --- R3 : completude des colonnes critiques ----------------------------
    manquantes = []
    for colonne in COLONNES_CRITIQUES:
        if colonne not in df.columns:
            resultat.erreurs.append(f"R3 completude : colonne absente '{colonne}'")
            continue
        nb_nan = int(df[colonne].isna().sum())
        if nb_nan:
            manquantes.append(f"{colonne} ({nb_nan:,})")
    if manquantes:
        resultat.erreurs.append(
            f"R3 completude : valeurs manquantes sur {', '.join(manquantes)}"
        )
    else:
        resultat.controles.append(
            f"R3 completude : {len(COLONNES_CRITIQUES)} colonnes critiques sans valeur manquante"
        )

    # --- R4 : domaines de valeurs ------------------------------------------
    grades_inconnus = set(df["qualite"].dropna().unique()) - GRADES_ATTENDUS
    if grades_inconnus:
        resultat.erreurs.append(
            f"R4 domaine : grade(s) inconnu(s) {sorted(grades_inconnus)}. "
            f"Attendus : {sorted(GRADES_ATTENDUS)}"
        )
    else:
        resultat.controles.append("R4 domaine : tous les grades appartiennent au referentiel")

    if cooperatives is not None:
        regions_connues = set(cooperatives["region"])
        regions_inconnues = set(df["region"].dropna().unique()) - regions_connues
        if regions_inconnues:
            resultat.erreurs.append(
                f"R4 domaine : region(s) non normalisee(s) {sorted(regions_inconnues)[:5]}"
            )
        else:
            resultat.controles.append(
                f"R4 domaine : {df['region'].nunique()} regions, toutes normalisees"
            )

    # --- R5 : bornes metier -------------------------------------------------
    hors_bornes = []
    for colonne, (mini, maxi) in BORNES.items():
        if colonne not in df.columns:
            continue
        serie = df[colonne].dropna()
        nb = int(((serie < mini) | (serie > maxi)).sum())
        if nb:
            hors_bornes.append(f"{colonne} ({nb:,} hors [{mini:g} ; {maxi:g}])")
    if hors_bornes:
        resultat.erreurs.append(f"R5 bornes : {', '.join(hors_bornes)}")
    else:
        resultat.controles.append("R5 bornes : tonnage, prix et humidite dans les plages metier")

    # --- R6 : integrite referentielle --------------------------------------
    if planteurs is not None:
        orphelins = int((~df["id_planteur"].isin(planteurs["id_planteur"])).sum())
        if orphelins:
            resultat.erreurs.append(
                f"R6 integrite : {orphelins:,} pesee(s) referencent un planteur absent "
                "du referentiel"
            )
        else:
            resultat.controles.append("R6 integrite : tous les planteurs existent au referentiel")

    if cooperatives is not None:
        orphelines = int((~df["cooperative"].isin(cooperatives["code_cooperative"])).sum())
        if orphelines:
            resultat.erreurs.append(
                f"R6 integrite : {orphelines:,} pesee(s) referencent une cooperative inconnue"
            )
        else:
            resultat.controles.append(
                "R6 integrite : toutes les cooperatives existent au referentiel"
            )

    # --- R7 : coherence hierarchique ---------------------------------------
    # C'est le garde-fou du schema en etoile : la table de faits portera a la
    # fois la region et la cooperative, rien n'interdit techniquement qu'elles
    # se contredisent. Cette regle verifie qu'elles ne le font pas.
    if planteurs is not None and "region" in df.columns:
        reference = planteurs.set_index("id_planteur")["region"]
        attendue = df["id_planteur"].map(reference)
        incoherentes = int((df["region"] != attendue).sum())
        if incoherentes:
            resultat.erreurs.append(
                f"R7 coherence : {incoherentes:,} pesee(s) dont la region differe de celle "
                "du planteur au referentiel"
            )
        else:
            resultat.controles.append(
                "R7 coherence : region de la pesee identique a celle du planteur"
            )

    # --- Avertissements non bloquants --------------------------------------
    for drapeau, libelle in (("prix_impute", "prix"), ("humidite_imputee", "humidite")):
        if drapeau in df.columns:
            part = df[drapeau].mean() * 100
            if part > 5:
                resultat.avertissements.append(
                    f"{part:.1f} % des lignes ont une valeur de {libelle} imputee, "
                    "prudence dans l'interpretation"
                )

    return resultat


def detecter_anomalies(df: pd.DataFrame) -> pd.DataFrame:
    """
    Detecte les anomalies de pesee sans supprimer aucune ligne.

    Trois familles :
      - tonnage hors de l'intervalle interquartile elargi (methode de Tukey)
      - humidite au-dessus de la norme d'exportation de 8 %
      - prix eloigne de la mediane de son grade de plus de 30 %

    Le resultat est un recapitulatif : combien de lignes, quelle part, quel
    tonnage concerne. C'est le materiau du rapport de qualite des donnees
    demande par le sujet.
    """
    lignes = []

    # --- Tonnages atypiques : methode de Tukey ------------------------------
    # Q1 et Q3 sont les quartiles, l'ecart interquartile mesure la dispersion
    # centrale. Au-dela de 3 ecarts, la valeur est consideree extreme.
    q1, q3 = df["tonnage_kg"].quantile([0.25, 0.75])
    ecart = q3 - q1
    plafond = q3 + 3 * ecart
    masque = df["tonnage_kg"] > plafond
    lignes.append(
        {
            "anomalie": "tonnage extreme",
            "critere": f"> {plafond:,.0f} kg (Q3 + 3 x IQR)",
            "nb_lignes": int(masque.sum()),
            "part_pct": round(masque.mean() * 100, 2),
            "tonnage_concerne_kg": round(float(df.loc[masque, "tonnage_kg"].sum()), 1),
            "action": "signale, conserve",
        }
    )

    # --- Humidite au-dessus de la norme d'export ---------------------------
    masque = df["humidite_pct"] > HUMIDITE_NORME_EXPORT
    lignes.append(
        {
            "anomalie": "humidite hors norme export",
            "critere": f"> {HUMIDITE_NORME_EXPORT} %",
            "nb_lignes": int(masque.sum()),
            "part_pct": round(masque.mean() * 100, 2),
            "tonnage_concerne_kg": round(float(df.loc[masque, "tonnage_kg"].sum()), 1),
            "action": "signale, conserve",
        }
    )

    # --- Prix eloigne de la mediane de son grade ---------------------------
    mediane_grade = df.groupby("qualite")["prix_fcfa_kg"].transform("median")
    ecart_relatif = (df["prix_fcfa_kg"] - mediane_grade).abs() / mediane_grade
    masque = ecart_relatif > 0.30
    lignes.append(
        {
            "anomalie": "prix atypique pour le grade",
            "critere": "ecart > 30 % a la mediane du grade",
            "nb_lignes": int(masque.sum()),
            "part_pct": round(masque.mean() * 100, 2),
            "tonnage_concerne_kg": round(float(df.loc[masque, "tonnage_kg"].sum()), 1),
            "action": "signale, conserve",
        }
    )

    return pd.DataFrame(lignes)


def taux_manquants(df: pd.DataFrame) -> pd.DataFrame:
    """Part de valeurs manquantes par colonne, pour le rapport de qualite."""
    manquants = df.isna().sum()
    tableau = pd.DataFrame(
        {
            "colonne": manquants.index,
            "nb_manquants": manquants.to_numpy(),
            "part_pct": (manquants / len(df) * 100).round(2).to_numpy(),
        }
    )
    return tableau[tableau["nb_manquants"] > 0].reset_index(drop=True)

In [30]:
# ---------------------------------------------------------------------------
# Application des regles et detection des anomalies
# ---------------------------------------------------------------------------
titre("4. Controle qualite")

resultat = valider(pesees, planteurs=planteurs, cooperatives=cooperatives,
                   seuil_lignes=max(1, int(len(pesees) * 0.9)))

lignes_controle = (
    [{"niveau": "OK", "message": c} for c in resultat.controles]
    + [{"niveau": "AVERTISSEMENT", "message": a} for a in resultat.avertissements]
    + [{"niveau": "ERREUR", "message": e} for e in resultat.erreurs]
)
afficher_df(pd.DataFrame(lignes_controle), "Regles de validation", max_lignes=20)

if resultat.est_valide:
    ok(resultat.resume())
else:
    echec(resultat.resume())

anomalies = detecter_anomalies(pesees)
afficher_df(anomalies, "Anomalies de pesee (signalees, non supprimees)")

╭─────────────────────╮
│ 4. Controle qualite │
╰─────────────────────╯

                            Regles de validation                            
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ niveau ┃ message                                                         ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ OK     │ R1 volume : 78,800 lignes (seuil 70,920)                        │
│ OK     │ R2 unicite : aucun doublon sur id_pesee                         │
│ OK     │ R3 completude : 8 colonnes critiques sans valeur manquante      │
│ OK     │ R4 domaine : tous les grades appartiennent au referentiel       │
│ OK     │ R4 domaine : 8 regions, toutes normalisees                      │
│ OK     │ R5 bornes : tonnage, prix et humidite dans les plages metier    │
│ OK     │ R6 integrite : tous les planteurs existent au referentiel       │
│ OK     │ R6 integrite : toutes les cooperatives existent au referentiel  │
│ OK     │ R7 coherence : region de la pesee identique a celle du planteur │
└────────┴─────────────────────────────────────────────────────────────────┘

OK QUALITE OK : 9 regle(s) passee(s), 0 erreur(s), 0 avertissement(s) sur 78,800 lignes

                                  Anomalies de pesee (signalees, non supprimees)                                   
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ anomalie              ┃ critere                ┃ nb_lignes ┃ part_pct ┃ tonnage_concerne_kg ┃ action            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ tonnage extreme       │ > 1,236 kg (Q3 + 3 x   │       554 │      0.7 │           818,774.4 │ signale, conserve │
│                       │ IQR)                   │           │          │                     │                   │
│ humidite hors norme   │ > 8.0 %                │    23,272 │    29.53 │         7,201,095.7 │ signale, conserve │
│ export                │                        │           │          │                     │                   │
│ prix atypique pour le │ ecart > 30 % a la      │        21 │     0.03 │             8,873.4 │ signale, conserve │
│ grade                 │ mediane du grade       │           │          │                     │                   │
└───────────────────────┴────────────────────────┴───────────┴──────────┴─────────────────────┴───────────────────┘

## 5. Schéma en étoile

Cinq dimensions et une table de faits. La table de faits ne garde que des nombres, tout
le descriptif part dans les dimensions.

**Convention de nommage** : `id_xxx` désigne une clé technique entière générée par
PostgreSQL, `code_xxx` le code métier issu de la source. La source appelle `id_planteur`
un code au format `PLT00042` ; dans `dim_planteur`, ce code devient `code_planteur`.

**Étoile plutôt que flocon** : la table de faits porte directement les trois clés
`id_region`, `id_cooperative` et `id_planteur`, malgré la hiérarchie naturelle entre les
trois. La redondance est assumée en échange d'une jointure unique pour l'analyse par
région, et la cohérence est garantie par la règle R7.

`id_pesee` reste clé primaire de la table de faits : c'est une dimension dégénérée, un
identifiant métier sans attribut propre, qui assure la traçabilité jusqu'au ticket
d'origine.

In [31]:
# ---------------------------------------------------------------------------
# Bareme de qualite.
# En production, ce referentiel viendrait d'un fichier officiel du Conseil du
# Cafe-Cacao. Il est reproduit ici a l'identique de celui utilise par le
# generateur : humidite maximale toleree et bornes de prix par grade.
# ---------------------------------------------------------------------------
REFERENTIEL_QUALITES = [
    # nom          rang  humidite_max  plancher  plafond  exportable
    ("Grade A",    1,    7.0,          900,      1100,    True),
    ("Grade B",    2,    8.0,          750,       900,    True),
    ("Grade C",    3,    9.0,          600,       750,    True),
    ("Hors grade", 4,   99.0,          400,       600,    False),
]

NOMS_MOIS = {
    1: "Janvier", 2: "Fevrier", 3: "Mars", 4: "Avril", 5: "Mai", 6: "Juin",
    7: "Juillet", 8: "Aout", 9: "Septembre", 10: "Octobre", 11: "Novembre",
    12: "Decembre",
}
NOMS_JOURS = {
    0: "Lundi", 1: "Mardi", 2: "Mercredi", 3: "Jeudi",
    4: "Vendredi", 5: "Samedi", 6: "Dimanche",
}
MOIS_CAMPAGNE_PRINCIPALE = {10, 11, 12, 1, 2, 3}

CHUNK_SIZE = 5_000

# Ordre de chargement, impose par les cles etrangeres
ORDRE_DIMENSIONS = ["dim_region", "dim_cooperative", "dim_planteur", "dim_qualite", "dim_date"]
TOUTES_LES_TABLES = ["faits_pesees", *ORDRE_DIMENSIONS]

CLES_FAITS = ["id_date", "id_region", "id_cooperative", "id_planteur", "id_qualite"]


# ---------------------------------------------------------------------------
# Construction des dimensions (aucune connexion a la base a ce stade)
# ---------------------------------------------------------------------------
def construire_dim_region(cooperatives: pd.DataFrame, planteurs: pd.DataFrame) -> pd.DataFrame:
    """
    Une ligne par region, avec deux attributs calcules.

    Une dimension qui ne contient qu'un identifiant et un nom n'apporte rien :
    le nombre de cooperatives et de planteurs rattaches permet de relativiser
    les volumes de production region par region.
    """
    regions = (
        cooperatives.groupby("region")
        .agg(
            zone_production=("zone_production", "first"),
            port_export=("port_export", "first"),
            nb_cooperatives=("code_cooperative", "count"),
        )
        .reset_index()
        .rename(columns={"region": "nom_region"})
    )
    planteurs_par_region = planteurs.groupby("region").size()
    regions["nb_planteurs"] = regions["nom_region"].map(planteurs_par_region).astype(int)

    return regions.sort_values("nom_region").reset_index(drop=True)


def construire_dim_cooperative(cooperatives: pd.DataFrame, planteurs: pd.DataFrame) -> pd.DataFrame:
    """Une ligne par cooperative, avec son nombre d'adherents."""
    coops = cooperatives.copy()
    adherents = planteurs.groupby("code_cooperative").size()
    coops["nb_planteurs"] = coops["code_cooperative"].map(adherents).fillna(0).astype(int)

    colonnes = ["code_cooperative", "nom_cooperative", "region",
                "annee_creation", "certifiee_bio", "nb_planteurs"]
    return coops[colonnes].sort_values("code_cooperative").reset_index(drop=True)


def construire_dim_planteur(planteurs: pd.DataFrame) -> pd.DataFrame:
    """Une ligne par planteur. Le code metier devient code_planteur."""
    dim = planteurs.rename(columns={"id_planteur": "code_planteur"}).copy()
    colonnes = ["code_planteur", "code_cooperative", "region",
                "superficie_ha", "annee_adhesion", "certifie_bio"]
    return dim[colonnes].sort_values("code_planteur").reset_index(drop=True)


def construire_dim_qualite() -> pd.DataFrame:
    """Les quatre grades du bareme, avec leurs seuils et leurs bornes de prix."""
    return pd.DataFrame(
        REFERENTIEL_QUALITES,
        columns=["nom_qualite", "rang", "humidite_max_pct",
                 "prix_plancher", "prix_plafond", "exportable"],
    )


def construire_dim_date(pesees: pd.DataFrame) -> pd.DataFrame:
    """
    Calendrier continu couvrant la periode des pesees.

    Tous les jours y figurent, y compris ceux sans aucune pesee : c'est ce qui
    permet ensuite de reperer les creux d'activite. Un calendrier construit a
    partir des seules dates presentes dans les faits les rendrait invisibles.
    """
    debut = pd.Timestamp(pesees["date"].min()).normalize()
    fin = pd.Timestamp(pesees["date"].max()).normalize()
    jours = pd.date_range(debut, fin, freq="D")

    dim = pd.DataFrame(
        {
            "date_complete": jours.date,
            "annee": jours.year,
            "mois": jours.month,
            "nom_mois": jours.month.map(NOMS_MOIS),
            "trimestre": jours.quarter,
            "semaine": jours.isocalendar().week.astype(int),
            "jour": jours.day,
            "nom_jour": jours.dayofweek.map(NOMS_JOURS),
            "est_weekend": jours.dayofweek >= 5,
        }
    )

    # Campagne et saison : calcules une fois ici, jamais dans les requetes
    dim["campagne"] = np.where(
        dim["mois"].isin(MOIS_CAMPAGNE_PRINCIPALE), "Principale", "Intermediaire"
    )
    annee_saison = np.where(dim["mois"] >= 10, dim["annee"], dim["annee"] - 1)
    dim["saison"] = [f"{a}-{a + 1}" for a in annee_saison]

    # Rang du mois dans la campagne : octobre = 1, septembre = 12.
    # Sans cette colonne, un tri par mois calendaire placerait janvier avant
    # octobre a l'interieur d'une meme saison, ce qui fausse toute comparaison
    # d'un mois au precedent.
    dim["mois_campagne"] = ((dim["mois"] - 10) % 12) + 1

    return dim


def construire_dimensions(
    pesees: pd.DataFrame, planteurs: pd.DataFrame, cooperatives: pd.DataFrame
) -> dict[str, pd.DataFrame]:
    """Assemble les cinq dimensions dans l'ordre de chargement."""
    etape("Construction des dimensions")

    dimensions = {
        "dim_region": construire_dim_region(cooperatives, planteurs),
        "dim_cooperative": construire_dim_cooperative(cooperatives, planteurs),
        "dim_planteur": construire_dim_planteur(planteurs),
        "dim_qualite": construire_dim_qualite(),
        "dim_date": construire_dim_date(pesees),
    }
    for nom, dim in dimensions.items():
        console.print(f"  {nom:<18} : {len(dim):>5,} lignes x {len(dim.columns)} colonnes")

    return dimensions


# ---------------------------------------------------------------------------
# Resolution des cles etrangeres a l'interieur des dimensions
# ---------------------------------------------------------------------------
def resoudre_cles_dimensions(
    dimensions: dict[str, pd.DataFrame], lookups: dict[str, dict]
) -> dict[str, pd.DataFrame]:
    """
    Remplace les libelles par les cles techniques dans dim_cooperative et
    dim_planteur, qui portent la region et la cooperative de rattachement.
    """
    coops = dimensions["dim_cooperative"].copy()
    coops["id_region"] = coops["region"].map(lookups["region"])
    dimensions["dim_cooperative"] = coops.drop(columns="region")

    planteurs = dimensions["dim_planteur"].copy()
    planteurs["id_region"] = planteurs["region"].map(lookups["region"])
    planteurs["id_cooperative"] = planteurs["code_cooperative"].map(lookups["cooperative"])
    dimensions["dim_planteur"] = planteurs.drop(columns=["region", "code_cooperative"])

    return dimensions


def construire_faits(pesees: pd.DataFrame, lookups: dict[str, dict]) -> pd.DataFrame:
    """
    Remplace chaque libelle de la pesee par la cle technique de sa dimension.

    Le controle qui suit est essentiel : une cle non resolue signifie qu'un
    libelle n'existe pas dans la dimension correspondante. Supprimer ces lignes
    sans les compter ferait disparaitre du tonnage en silence.
    """
    etape("Construction de la table de faits")

    faits = pd.DataFrame(
        {
            "id_pesee": pesees["id_pesee"],
            "id_date": pd.to_datetime(pesees["date"]).dt.date.map(lookups["date"]),
            "id_region": pesees["region"].map(lookups["region"]),
            "id_cooperative": pesees["cooperative"].map(lookups["cooperative"]),
            "id_planteur": pesees["id_planteur"].map(lookups["planteur"]),
            "id_qualite": pesees["qualite"].map(lookups["qualite"]),
            "tonnage_kg": pesees["tonnage_kg"],
            "montant_fcfa": pesees["montant_fcfa"],
            "prix_fcfa_kg": pesees["prix_fcfa_kg"],
            "humidite_pct": pesees["humidite_pct"],
            "ecart_prix_grade_pct": pesees["ecart_prix_grade_pct"],
            "categorie_tonnage": pesees["categorie_tonnage"],
            "conforme_export": pesees["conforme_export"],
            "prix_impute": pesees["prix_impute"],
            "humidite_imputee": pesees["humidite_imputee"],
        }
    )

    non_resolues = faits[faits[CLES_FAITS].isna().any(axis=1)]
    if non_resolues.empty:
        ok(f"{len(faits):,} lignes, toutes les cles etrangeres sont resolues")
    else:
        alerte(f"{len(non_resolues):,} ligne(s) sans correspondance, elles seront ecartees")
        detail = (
            non_resolues[CLES_FAITS].isna().sum().to_frame("lignes_sans_cle").reset_index()
        )
        detail.columns = ["cle_etrangere", "lignes_sans_cle"]
        afficher_df(detail, "Detail des cles non resolues")
        faits = faits.dropna(subset=CLES_FAITS)

    faits[CLES_FAITS] = faits[CLES_FAITS].astype(int)
    return faits


# ---------------------------------------------------------------------------
# Chargement dans Supabase
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Mode sans base
# ---------------------------------------------------------------------------
def simuler_lookups(dimensions: dict[str, pd.DataFrame]) -> dict[str, dict]:
    """
    Attribue les cles techniques comme le ferait PostgreSQL : 1, 2, 3...

    Sert uniquement au mode --dry-run, pour valider la logique de resolution
    des cles etrangeres sans connexion a la base.
    """
    return {
        "region": {v: i for i, v in enumerate(dimensions["dim_region"]["nom_region"], 1)},
        "cooperative": {
            v: i for i, v in enumerate(dimensions["dim_cooperative"]["code_cooperative"], 1)
        },
        "planteur": {
            v: i for i, v in enumerate(dimensions["dim_planteur"]["code_planteur"], 1)
        },
        "qualite": {v: i for i, v in enumerate(dimensions["dim_qualite"]["nom_qualite"], 1)},
        "date": {v: i for i, v in enumerate(dimensions["dim_date"]["date_complete"], 1)},
    }


def ajouter_cles_simulees(
    dimensions: dict[str, pd.DataFrame], lookups: dict[str, dict]
) -> dict[str, pd.DataFrame]:
    """
    Ajoute la colonne de cle technique aux dimensions, en mode --dry-run.

    Sans elle, la copie Parquet ne refleterait pas ce que contient la base :
    on ne pourrait pas rejouer les requetes analytiques en local sur ces
    fichiers pour verifier une jointure.
    """
    correspondances = {
        "dim_region": ("id_region", "nom_region", "region"),
        "dim_cooperative": ("id_cooperative", "code_cooperative", "cooperative"),
        "dim_planteur": ("id_planteur", "code_planteur", "planteur"),
        "dim_qualite": ("id_qualite", "nom_qualite", "qualite"),
        "dim_date": ("id_date", "date_complete", "date"),
    }
    for table, (colonne_cle, colonne_code, cle_lookup) in correspondances.items():
        dim = dimensions[table].copy()
        dim.insert(0, colonne_cle, dim[colonne_code].map(lookups[cle_lookup]).astype(int))
        dimensions[table] = dim
    return dimensions

In [32]:
# ---------------------------------------------------------------------------
# Construction des dimensions et de la table de faits
# ---------------------------------------------------------------------------
titre("5. Schema en etoile")

dimensions = construire_dimensions(pesees, planteurs, cooperatives)

# Les cles techniques sont simulees ici comme PostgreSQL les attribuerait : 1, 2, 3...
lookups = simuler_lookups(dimensions)
dimensions = resoudre_cles_dimensions(dimensions, lookups)
faits = construire_faits(pesees, lookups)
dimensions = ajouter_cles_simulees(dimensions, lookups)

for nom, dim in dimensions.items():
    sauver_etape(dim, f"etoile_{nom}")
sauver_etape(faits, "etoile_faits_pesees")

afficher_df(
    pd.DataFrame([{"table": n, "lignes": len(d), "colonnes": len(d.columns)}
                  for n, d in {**dimensions, "faits_pesees": faits}.items()]),
    "Tables du schema en etoile",
)

╭─────────────────────╮
│ 5. Schema en etoile │
╰─────────────────────╯

> Construction des dimensions

dim_region         :     8 lignes x 5 colonnes

dim_cooperative    :    40 lignes x 6 colonnes

dim_planteur       : 5,000 lignes x 6 colonnes

dim_qualite        :     4 lignes x 6 colonnes

dim_date           :   731 lignes x 12 colonnes

> Construction de la table de faits

OK 78,800 lignes, toutes les cles etrangeres sont resolues

      Tables du schema en etoile       
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┓
┃ table           ┃ lignes ┃ colonnes ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━┩
│ dim_region      │      8 │        6 │
│ dim_cooperative │     40 │        7 │
│ dim_planteur    │   5000 │        7 │
│ dim_qualite     │      4 │        7 │
│ dim_date        │    731 │       13 │
│ faits_pesees    │ 78,800 │       15 │
└─────────────────┴────────┴──────────┘

### Chargement dans Supabase

Le DDL ci-dessous est celui du fichier `sql/01_creer_schema_etoile.sql` du dépôt.

Cette cellule ne s'exécute que si la variable d'environnement `SUPABASE_URL` est
renseignée. Sans elle, le notebook continue sans erreur : les tables restent en local et
les requêtes de la partie suivante s'exécuteront sur DuckDB.

In [33]:
# ---------------------------------------------------------------------------
# DDL du schema en etoile (identique a sql/01_creer_schema_etoile.sql)
# ---------------------------------------------------------------------------
DDL_ETOILE = """
CREATE TABLE IF NOT EXISTS dim_region (
    id_region        SERIAL PRIMARY KEY,
    nom_region       TEXT NOT NULL UNIQUE,
    zone_production  TEXT,
    port_export      TEXT,
    nb_cooperatives  INTEGER,
    nb_planteurs     INTEGER
);

CREATE TABLE IF NOT EXISTS dim_cooperative (
    id_cooperative   SERIAL PRIMARY KEY,
    code_cooperative TEXT NOT NULL UNIQUE,
    nom_cooperative  TEXT,
    id_region        INTEGER NOT NULL REFERENCES dim_region(id_region),
    annee_creation   INTEGER,
    certifiee_bio    BOOLEAN DEFAULT FALSE,
    nb_planteurs     INTEGER
);

CREATE TABLE IF NOT EXISTS dim_planteur (
    id_planteur      SERIAL PRIMARY KEY,
    code_planteur    TEXT NOT NULL UNIQUE,
    id_cooperative   INTEGER NOT NULL REFERENCES dim_cooperative(id_cooperative),
    id_region        INTEGER NOT NULL REFERENCES dim_region(id_region),
    superficie_ha    NUMERIC(5,1),
    annee_adhesion   INTEGER,
    certifie_bio     BOOLEAN DEFAULT FALSE
);

CREATE TABLE IF NOT EXISTS dim_qualite (
    id_qualite       SERIAL PRIMARY KEY,
    nom_qualite      TEXT NOT NULL UNIQUE,
    rang             INTEGER,
    humidite_max_pct NUMERIC(4,1),
    prix_plancher    INTEGER,
    prix_plafond     INTEGER,
    exportable       BOOLEAN
);

CREATE TABLE IF NOT EXISTS dim_date (
    id_date        SERIAL PRIMARY KEY,
    date_complete  DATE NOT NULL UNIQUE,
    annee          INTEGER,
    mois           INTEGER,
    mois_campagne  INTEGER,
    nom_mois       TEXT,
    trimestre      INTEGER,
    semaine        INTEGER,
    jour           INTEGER,
    nom_jour       TEXT,
    est_weekend    BOOLEAN,
    campagne       TEXT,
    saison         TEXT
);

CREATE TABLE IF NOT EXISTS faits_pesees (
    id_pesee             TEXT PRIMARY KEY,
    id_date              INTEGER NOT NULL REFERENCES dim_date(id_date),
    id_region            INTEGER NOT NULL REFERENCES dim_region(id_region),
    id_cooperative       INTEGER NOT NULL REFERENCES dim_cooperative(id_cooperative),
    id_planteur          INTEGER NOT NULL REFERENCES dim_planteur(id_planteur),
    id_qualite           INTEGER NOT NULL REFERENCES dim_qualite(id_qualite),
    tonnage_kg           NUMERIC(10,1) NOT NULL,
    montant_fcfa         BIGINT NOT NULL,
    prix_fcfa_kg         INTEGER NOT NULL,
    humidite_pct         NUMERIC(4,1),
    ecart_prix_grade_pct NUMERIC(6,1),
    categorie_tonnage    TEXT,
    conforme_export      BOOLEAN,
    prix_impute          BOOLEAN DEFAULT FALSE,
    humidite_imputee     BOOLEAN DEFAULT FALSE
);

CREATE INDEX IF NOT EXISTS idx_faits_date   ON faits_pesees(id_date);
CREATE INDEX IF NOT EXISTS idx_faits_region ON faits_pesees(id_region);
"""

engine = get_engine()

if engine is None:
    alerte("SUPABASE_URL absente : chargement ignore, les requetes s'executeront en local.")
else:
    from sqlalchemy import text

    with engine.begin() as conn:
        conn.execute(text(DDL_ETOILE))
        # TRUNCATE refuse de vider une table referencee par une autre, sauf si
        # cette autre est citee dans la meme instruction : on les liste toutes.
        conn.execute(text(
            "TRUNCATE faits_pesees, dim_region, dim_cooperative, "
            "dim_planteur, dim_qualite, dim_date RESTART IDENTITY"
        ))
        # Les colonnes id_* simulees sont retirees : PostgreSQL les regenere
        for nom_table, dim in dimensions.items():
            colonne_cle = nom_table.replace("dim_", "id_")
            dim.drop(columns=[colonne_cle], errors="ignore").to_sql(
                nom_table, conn, if_exists="append", index=False,
                method="multi", chunksize=1_000,
            )
        faits.to_sql("faits_pesees", conn, if_exists="append", index=False,
                     method="multi", chunksize=1_000)
    ok("Schema en etoile charge dans Supabase")

!  SUPABASE_URL absente : chargement ignore, les requetes s'executeront en local.

## 6. Requêtes analytiques

Six requêtes, dont une avec CTE et fonctions de fenêtre. Deux conventions les traversent.

**Cast avant `ROUND`.** En PostgreSQL, `ROUND(valeur, décimales)` n'accepte pas un
`double precision`, seulement un `numeric`. Dès qu'une division produit un flottant,
l'appel échoue avec un message peu explicite.

**Moyenne pondérée pour les prix.** Le prix est une mesure non additive :
`AVG(prix_fcfa_kg)` donne le même poids à une pesée de 12 kg et à une pesée de 2 000 kg.
Le prix réellement payé est `SUM(montant_fcfa) / SUM(tonnage_kg)`.

Un détail qui a coûté un débogage : trier par saison puis par mois calendaire place
janvier avant octobre à l'intérieur d'une même saison, et `LAG` comparait donc chaque
mois au mauvais mois précédent. D'où la colonne `mois_campagne` dans `dim_date`, valant
1 pour octobre et 12 pour septembre.

In [34]:
# ---------------------------------------------------------------------------
# REQUETE 1 : production par region
# Jointure simple entre la table de faits et dim_region.
# Question metier : quelles regions portent la production, et le prix paye
# y varie-t-il ?
# ---------------------------------------------------------------------------
Q1_PRODUCTION_REGION = """
SELECT
    r.nom_region,
    r.zone_production,
    r.port_export,
    COUNT(*)                                        AS nb_pesees,
    ROUND((SUM(f.tonnage_kg) / 1000)::numeric, 1)   AS tonnes,
    ROUND((100.0 * SUM(f.tonnage_kg)
           / SUM(SUM(f.tonnage_kg)) OVER ())::numeric, 1) AS part_nationale_pct,
    ROUND((SUM(f.montant_fcfa) / SUM(f.tonnage_kg))::numeric, 0) AS prix_moyen_pondere,
    ROUND((SUM(f.montant_fcfa) / 1e9)::numeric, 2)  AS valeur_milliards_fcfa
FROM faits_pesees f
JOIN dim_region r ON f.id_region = r.id_region
GROUP BY r.nom_region, r.zone_production, r.port_export
ORDER BY tonnes DESC
"""

# ---------------------------------------------------------------------------
# REQUETE 2 : prix et conformite par qualite
# Jointure avec dim_qualite. Les deux moyennes sont affichees cote a cote
# pour montrer l'ecart entre une moyenne simple et une moyenne ponderee.
# Le seuil d'humidite vient de la dimension, il n'est ecrit nulle part en dur.
# ---------------------------------------------------------------------------
Q2_PRIX_QUALITE = """
SELECT
    q.nom_qualite,
    q.rang,
    q.humidite_max_pct,
    q.exportable,
    COUNT(*)                                        AS nb_pesees,
    ROUND((SUM(f.tonnage_kg) / 1000)::numeric, 1)   AS tonnes,
    ROUND(AVG(f.prix_fcfa_kg)::numeric, 0)          AS prix_moyen_simple,
    ROUND((SUM(f.montant_fcfa) / SUM(f.tonnage_kg))::numeric, 0) AS prix_moyen_pondere,
    ROUND(AVG(f.humidite_pct)::numeric, 2)          AS humidite_moyenne,
    ROUND((100.0 * COUNT(*) FILTER (WHERE f.conforme_export)
           / COUNT(*))::numeric, 1)                 AS conforme_export_pct
FROM faits_pesees f
JOIN dim_qualite q ON f.id_qualite = q.id_qualite
GROUP BY q.nom_qualite, q.rang, q.humidite_max_pct, q.exportable
ORDER BY q.rang
"""

# ---------------------------------------------------------------------------
# REQUETE 3 : saisonnalite mensuelle
# Jointure avec dim_date. Aucun calcul de date dans la requete : la campagne
# et la saison sont des colonnes de la dimension, calculees une fois au
# chargement. C'est tout l'interet d'une dimension calendrier.
# ---------------------------------------------------------------------------
Q3_SAISONNALITE = """
SELECT
    d.saison,
    d.mois_campagne,          -- 1 = octobre, ordre chronologique de la campagne
    d.nom_mois,
    d.campagne,
    COUNT(*)                                        AS nb_pesees,
    ROUND((SUM(f.tonnage_kg) / 1000)::numeric, 1)   AS tonnes,
    ROUND((SUM(f.montant_fcfa) / SUM(f.tonnage_kg))::numeric, 0) AS prix_moyen_pondere,
    ROUND((100.0 * COUNT(*) FILTER (WHERE q.nom_qualite = 'Grade A')
           / COUNT(*))::numeric, 1)                 AS part_grade_a_pct
FROM faits_pesees f
JOIN dim_date d    ON f.id_date    = d.id_date
JOIN dim_qualite q ON f.id_qualite = q.id_qualite
GROUP BY d.saison, d.mois_campagne, d.nom_mois, d.campagne
ORDER BY d.saison, d.mois_campagne
"""

# ---------------------------------------------------------------------------
# REQUETE 4 : classement des cooperatives
# Jointure triple : faits, cooperative, region. La certification bio vient de
# dim_planteur, ce qui oblige une quatrieme jointure : c'est le prix a payer
# pour avoir range l'attribut au bon endroit.
# ---------------------------------------------------------------------------
Q4_COOPERATIVES = """
SELECT
    c.code_cooperative,
    r.nom_region,
    c.nb_planteurs,
    COUNT(*)                                        AS nb_pesees,
    ROUND((SUM(f.tonnage_kg) / 1000)::numeric, 1)   AS tonnes,
    ROUND((SUM(f.tonnage_kg) / c.nb_planteurs)::numeric, 0) AS kg_par_planteur,
    ROUND((SUM(f.montant_fcfa) / SUM(f.tonnage_kg))::numeric, 0) AS prix_moyen_pondere,
    ROUND((100.0 * COUNT(*) FILTER (WHERE p.certifie_bio)
           / COUNT(*))::numeric, 1)                 AS part_bio_pct
FROM faits_pesees f
JOIN dim_cooperative c ON f.id_cooperative = c.id_cooperative
JOIN dim_region r      ON c.id_region      = r.id_region
JOIN dim_planteur p    ON f.id_planteur    = p.id_planteur
GROUP BY c.code_cooperative, r.nom_region, c.nb_planteurs
ORDER BY tonnes DESC
LIMIT 15
"""

# ---------------------------------------------------------------------------
# REQUETE 5 : requete avancee, CTE et fonctions de fenetre
#
# Trois fonctions de fenetre y travaillent ensemble :
#   RANK() OVER (PARTITION BY mois ...)  classe les regions dans chaque mois
#   LAG() OVER (PARTITION BY region ...) va chercher le mois precedent
#   SUM() OVER (PARTITION BY mois)       donne le total du mois, denominateur
#                                        de la part de marche
#
# La CTE nomme le calcul intermediaire : sans elle, il faudrait repeter
# l'agregation dans chaque sous-requete.
# ---------------------------------------------------------------------------
Q5_CLASSEMENT_MENSUEL = """
WITH volume_mensuel AS (
    SELECT
        d.saison,
        d.mois_campagne,
        d.nom_mois,
        r.nom_region,
        SUM(f.tonnage_kg)  AS tonnage,
        SUM(f.montant_fcfa) AS montant
    FROM faits_pesees f
    JOIN dim_date d   ON f.id_date   = d.id_date
    JOIN dim_region r ON f.id_region = r.id_region
    GROUP BY d.saison, d.mois_campagne, d.nom_mois, r.nom_region
)
SELECT
    saison,
    mois_campagne,
    nom_mois,
    nom_region,
    ROUND((tonnage / 1000)::numeric, 1) AS tonnes,

    -- Rang de la region a l'interieur de chaque mois
    RANK() OVER (PARTITION BY saison, mois_campagne ORDER BY tonnage DESC) AS rang_mensuel,

    -- Part du mois portee par la region
    ROUND((100.0 * tonnage
           / SUM(tonnage) OVER (PARTITION BY saison, mois_campagne))::numeric, 1) AS part_du_mois_pct,

    -- Tonnage du mois precedent pour la meme region
    ROUND((LAG(tonnage) OVER (PARTITION BY nom_region ORDER BY saison, mois_campagne)
           / 1000)::numeric, 1) AS tonnes_mois_precedent,

    -- Variation par rapport au mois precedent
    -- NULLIF evite la division par zero quand le mois precedent est vide
    ROUND((100.0 * (tonnage - LAG(tonnage) OVER (PARTITION BY nom_region ORDER BY saison, mois_campagne))
           / NULLIF(LAG(tonnage) OVER (PARTITION BY nom_region ORDER BY saison, mois_campagne), 0)
          )::numeric, 1) AS variation_pct
FROM volume_mensuel
ORDER BY saison, mois_campagne, rang_mensuel
"""

# ---------------------------------------------------------------------------
# REQUETE 6 : prime bio, pour completer l'analyse economique
# ---------------------------------------------------------------------------
Q6_PRIME_BIO = """
SELECT
    q.nom_qualite,
    p.certifie_bio,
    COUNT(*)                                        AS nb_pesees,
    ROUND((SUM(f.tonnage_kg) / 1000)::numeric, 1)   AS tonnes,
    ROUND((SUM(f.montant_fcfa) / SUM(f.tonnage_kg))::numeric, 0) AS prix_moyen_pondere
FROM faits_pesees f
JOIN dim_qualite q  ON f.id_qualite  = q.id_qualite
JOIN dim_planteur p ON f.id_planteur = p.id_planteur
GROUP BY q.nom_qualite, q.rang, p.certifie_bio
ORDER BY q.rang, p.certifie_bio
"""


# Chaque entree : cle technique, requete, titre affiche, lecture metier
REQUETES = {
    "production_region": (
        Q1_PRODUCTION_REGION,
        "Requete 1 : production par region",
        "Ou se concentre la production, et le prix paye varie-t-il d'une region a l'autre ?",
    ),
    "prix_qualite": (
        Q2_PRIX_QUALITE,
        "Requete 2 : prix et conformite par qualite",
        "Combien vaut chaque grade, et quelle part respecte la norme d'exportation ?",
    ),
    "saisonnalite": (
        Q3_SAISONNALITE,
        "Requete 3 : saisonnalite mensuelle",
        "Comment les apports se repartissent-ils dans l'annee, et la qualite suit-elle ?",
    ),
    "cooperatives": (
        Q4_COOPERATIVES,
        "Requete 4 : classement des cooperatives",
        "Quelles cooperatives collectent le plus, et avec quelle productivite par planteur ?",
    ),
    "classement_mensuel": (
        Q5_CLASSEMENT_MENSUEL,
        "Requete 5 (avancee) : classement mensuel des regions",
        "Quelle region domine chaque mois, et comment son volume evolue-t-il ?",
    ),
    "prime_bio": (
        Q6_PRIME_BIO,
        "Requete 6 : effet de la certification bio",
        "La certification bio se traduit-elle par un meilleur prix a grade egal ?",
    ),
}

In [35]:
# ---------------------------------------------------------------------------
# Execution : sur Supabase si disponible, sinon en local avec DuckDB
# ---------------------------------------------------------------------------
titre("6. Requetes analytiques")

TABLES_ETOILE = ["dim_region", "dim_cooperative", "dim_planteur",
                 "dim_qualite", "dim_date", "faits_pesees"]

if engine is not None:
    etape("Execution sur Supabase")
    executer = lambda sql: pd.read_sql(sql, engine)
else:
    etape("Execution en local avec DuckDB")
    import duckdb

    connexion = duckdb.connect()
    for table in TABLES_ETOILE:
        chemin = DATA_INTERIM / f"etoile_{table}.parquet"
        connexion.execute(f"CREATE TABLE {table} AS SELECT * FROM read_parquet('{chemin}')")
    executer = lambda sql: connexion.execute(sql).df()

resultats = {}
for cle, (sql, titre_requete, lecture) in REQUETES.items():
    resultats[cle] = executer(sql)
    afficher_df(resultats[cle], titre_requete, max_lignes=10)
    console.print(f"  [dim]{lecture}[/]\n")
    sauver_etape(resultats[cle], f"resultat_{cle}")

╭─────────────────────────╮
│ 6. Requetes analytiques │
╰─────────────────────────╯

> Execution en local avec DuckDB

                                         Requete 1 : production par region                                         
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃            ┃ zone_producti ┃             ┃           ┃         ┃ part_national ┃ prix_moyen_po ┃ valeur_milliar ┃
┃ nom_region ┃ on            ┃ port_export ┃ nb_pesees ┃  tonnes ┃         e_pct ┃         ndere ┃        ds_fcfa ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ Soubre     │ Sud-Ouest     │ San Pedro   │    16,323 │ 5,142.6 │          21.1 │           850 │           4.37 │
│ San Pedro  │ Sud-Ouest     │ San Pedro   │    12,753 │ 3,916.8 │          16.1 │           853 │           3.34 │
│ Daloa      │ Centre-Ouest  │ San Pedro   │    10,951 │   3,326 │          13.7 │           850 │           2.83 │
│ Divo       │ Centre-Sud    │ Abidjan     │    10,092 │ 3,039.1 │          12.5 │           853 │           2.59 │
│ Gagnoa     │ Centre-Ouest  │ San Pedro   │      9022 │ 2,795.8 │          11.5 │           849 │           2.37 │
│ Abengourou │ Est           │ Abidjan     │      8787 │ 2,711.4 │          11.1 │           848 │            2.3 │
│ Aboisso    │ Sud-Est       │ Abidjan     │      7067 │ 2,238.5 │           9.2 │           857 │           1.92 │
│ Bondoukou  │ Nord-Est      │ Abidjan     │      3805 │   1,175 │           4.8 │           858 │           1.01 │
└────────────┴───────────────┴─────────────┴───────────┴─────────┴───────────────┴───────────────┴────────────────┘

Ou se concentre la production, et le prix paye varie-t-il d'une region a l'autre ?

                                    Requete 2 : prix et conformite par qualite                                     
┏━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃           ┃      ┃           ┃           ┃           ┃         ┃           ┃           ┃            ┃ conforme_ ┃
┃ nom_quali ┃      ┃ humidite_ ┃ exportabl ┃           ┃         ┃ prix_moye ┃ prix_moye ┃ humidite_m ┃ export_pc ┃
┃ te        ┃ rang ┃   max_pct ┃         e ┃ nb_pesees ┃  tonnes ┃  n_simple ┃ n_pondere ┃     oyenne ┃         t ┃
┡━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ Grade A   │    1 │         7 │       oui │    27,738 │ 8,593.5 │     1,006 │     1,017 │        6.6 │      99.8 │
│ Grade B   │    2 │         8 │       oui │    31,007 │ 9,504.5 │       828 │       837 │        7.6 │      82.5 │
│ Grade C   │    3 │         9 │       oui │    15,933 │ 4,956.7 │       675 │       682 │        8.6 │      13.3 │
│ Hors      │    4 │        99 │       non │      4122 │ 1,290.6 │       501 │       506 │      10.18 │       3.4 │
│ grade     │      │           │           │           │         │           │           │            │           │
└───────────┴──────┴───────────┴───────────┴───────────┴─────────┴───────────┴───────────┴────────────┴───────────┘

...

                               Requete 5 (avancee) : classement mensuel des regions                                
┏━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃           ┃             ┃          ┃            ┃        ┃             ┃             ┃ tonnes_moi ┃             ┃
┃           ┃ mois_campag ┃          ┃            ┃        ┃ rang_mensue ┃ part_du_moi ┃ s_preceden ┃ variation_p ┃
┃ saison    ┃          ne ┃ nom_mois ┃ nom_region ┃ tonnes ┃           l ┃       s_pct ┃          t ┃          ct ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ 2022-2023 │           1 │ Octobre  │ Soubre     │  399.4 │           1 │        20.6 │          - │           - │
│ 2022-2023 │           1 │ Octobre  │ San Pedro  │  320.8 │           2 │        16.5 │          - │           - │
│ 2022-2023 │           1 │ Octobre  │ Daloa      │  278.1 │           3 │        14.3 │          - │           - │
│ 2022-2023 │           1 │ Octobre  │ Gagnoa     │  238.7 │           4 │        12.3 │          - │           - │
│ 2022-2023 │           1 │ Octobre  │ Divo       │  233.4 │           5 │          12 │          - │           - │
│ 2022-2023 │           1 │ Octobre  │ Abengourou │  209.1 │           6 │        10.8 │          - │           - │
│ 2022-2023 │           1 │ Octobre  │ Aboisso    │  183.3 │           7 │         9.4 │          - │           - │
│ 2022-2023 │           1 │ Octobre  │ Bondoukou  │   80.1 │           8 │         4.1 │          - │           - │
│ 2022-2023 │           2 │ Novembre │ Soubre     │  472.9 │           1 │        21.4 │      399.4 │        18.4 │
│ 2022-2023 │           2 │ Novembre │ San Pedro  │  353.2 │           2 │          16 │      320.8 │        10.1 │
└───────────┴─────────────┴──────────┴────────────┴────────┴─────────────┴─────────────┴────────────┴─────────────┘

... 182 ligne(s) non affichee(s)

Quelle region domine chaque mois, et comment son volume evolue-t-il ?

                Requete 6 : effet de la certification bio                
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ nom_qualite ┃ certifie_bio ┃ nb_pesees ┃  tonnes ┃ prix_moyen_pondere ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ Grade A     │          non │    23,647 │ 7,350.3 │              1,003 │
│ Grade A     │          oui │      4091 │ 1,243.2 │              1,101 │
│ Grade B     │          non │    26,269 │ 8,056.8 │                825 │
│ Grade B     │          oui │      4738 │ 1,447.7 │                905 │
│ Grade C     │          non │    13,521 │ 4,235.5 │                673 │
│ Grade C     │          oui │      2412 │   721.2 │                738 │
│ Hors grade  │          non │      3481 │ 1,103.2 │                499 │
│ Hors grade  │          oui │       641 │   187.5 │                544 │
└─────────────┴──────────────┴───────────┴─────────┴────────────────────┘

La certification bio se traduit-elle par un meilleur prix a grade egal ?

### Lecture métier des résultats

**Production par région.** Soubré porte 21,1 % du tonnage, San Pedro 16,1 %, contre 4,8 %
pour Bondoukou. Les régions exportant par San Pedro pèsent près de 62 % du tonnage : une
saturation de ce port bloquerait la majorité de la filière.

**Prix par qualité.** Grade A à 1 017 FCFA/kg en moyenne pondérée contre 1 006 en moyenne
simple. La moyenne simple sous-estime le prix réellement payé.

**Conformité à l'exportation.** 99,8 % du Grade A respecte le seuil de 8 % d'humidité,
contre 13,3 % du Grade C. Près de 4 300 tonnes de Grade C sont invendables à l'export en
l'état : c'est le constat le plus actionnable du projet.

**Saisonnalité.** Pic à 2 209 tonnes en novembre, creux à 372 tonnes en août, soit un
rapport de 1 à 6. La part de Grade A passe de 38 % en campagne principale à 28 % en
campagne intermédiaire : le cacao séché pendant l'harmattan est meilleur, et le prix suit.

**Certification bio.** À grade égal, la prime est d'environ 10 %. C'est le levier de
revenu le plus direct pour un planteur, et il ne dépend pas de la qualité du séchage.

## 7. Tableau de bord

Six graphiques, dont les trois demandés par le sujet : cartographie des régions en
barres, évolution des prix, distribution des qualités.

In [36]:
matplotlib.use("Agg")   # backend sans fenetre, indispensable hors notebook


# Resolution imposee par le sujet : au moins 150 dpi
DPI = 150

# Palette inspiree de la filiere : bruns de la feve, vert du cacaoyer
BRUN_FONCE = "#4E342E"
BRUN = "#6D4C41"
OCRE = "#A1887F"
VERT = "#2E7D32"
OR = "#C8A415"
ROUGE = "#C62828"

COULEURS_GRADE = {
    "Grade A": VERT,
    "Grade B": OR,
    "Grade C": OCRE,
    "Hors grade": ROUGE,
}

HUMIDITE_NORME_EXPORT = 8.0

plt.rcParams.update(
    {
        "figure.facecolor": "#FAFAF7",
        "axes.facecolor": "#FFFFFF",
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.axisbelow": True,      # la grille passe derriere les barres
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def numeriser(df: pd.DataFrame, colonnes: list[str]) -> pd.DataFrame:
    """
    Convertit en flottants les colonnes issues du SQL.

    PostgreSQL renvoie les colonnes NUMERIC sous forme d'objets Decimal.
    Pandas les stocke alors en type object, et Matplotlib refuse de les
    tracer. La conversion est sans effet quand les donnees viennent deja de
    DuckDB, qui renvoie des flottants.
    """
    df = df.copy()
    for colonne in colonnes:
        if colonne in df.columns:
            df[colonne] = pd.to_numeric(df[colonne], errors="coerce")
    return df


def annoter_barres(ax, barres, valeurs, format_texte="{:.0f}", decalage=0.01,
                   horizontal=False) -> None:
    """
    Ecrit la valeur au bout de chaque barre.

    Un graphique doit se lire sans que l'oeil ait a estimer une hauteur sur
    l'axe : c'est un des criteres de lisibilite du bareme.
    """
    for barre, valeur in zip(barres, valeurs):
        if horizontal:
            largeur = barre.get_width()
            ax.text(
                largeur * (1 + decalage), barre.get_y() + barre.get_height() / 2,
                format_texte.format(valeur), va="center", fontsize=8,
            )
        else:
            hauteur = barre.get_height()
            ax.text(
                barre.get_x() + barre.get_width() / 2, hauteur * (1 + decalage),
                format_texte.format(valeur), ha="center", fontsize=8,
            )


# ---------------------------------------------------------------------------
# Graphique 1 : production par region
# ---------------------------------------------------------------------------
def graphique_production_region(ax) -> None:
    """Cartographie des regions en barres, demandee explicitement par le sujet."""
    df = numeriser(
        charger_etape("resultat_production_region"),
        ["tonnes", "part_nationale_pct", "prix_moyen_pondere"],
    ).sort_values("tonnes")

    barres = ax.barh(df["nom_region"], df["tonnes"], color=OCRE, alpha=0.9)

    # Les regions du Sud-Ouest, qui exportent par San Pedro, sont distinguees :
    # c'est la zone de production historique du cacao ivoirien.
    for barre, port in zip(barres, df["port_export"]):
        if port == "San Pedro":
            barre.set_color(BRUN_FONCE)

    annoter_barres(
        ax, barres, df["part_nationale_pct"], format_texte="{:.1f} %", horizontal=True
    )

    ax.set_title("Production par region (2 campagnes)")
    ax.set_xlabel("Tonnes collectees")
    ax.set_xlim(0, df["tonnes"].max() * 1.15)
    ax.legend(
        handles=[
            plt.Rectangle((0, 0), 1, 1, color=BRUN_FONCE),
            plt.Rectangle((0, 0), 1, 1, color=OCRE),
        ],
        labels=["Export via San Pedro", "Export via Abidjan"],
        fontsize=8,
        loc="lower right",
    )


# ---------------------------------------------------------------------------
# Graphique 2 : saisonnalite
# ---------------------------------------------------------------------------
def graphique_saisonnalite(ax) -> None:
    """
    Tonnage par mois de campagne, une serie par saison.

    L'axe suit l'ordre de la campagne, d'octobre a septembre, grace a la
    colonne mois_campagne de dim_date. Un axe en mois calendaire couperait la
    campagne principale en deux.
    """
    df = numeriser(charger_etape("resultat_saisonnalite"), ["tonnes", "mois_campagne"])

    pivot = df.pivot_table(
        index="mois_campagne", columns="saison", values="tonnes", aggfunc="sum"
    ).sort_index()
    etiquettes = (
        df.drop_duplicates("mois_campagne")
        .set_index("mois_campagne")["nom_mois"]
        .sort_index()
    )

    positions = np.arange(len(pivot))
    largeur = 0.38
    couleurs = [BRUN_FONCE, OCRE]

    for decalage, (saison, couleur) in enumerate(zip(pivot.columns, couleurs)):
        ax.bar(
            positions + (decalage - 0.5) * largeur,
            pivot[saison],
            width=largeur,
            label=saison,
            color=couleur,
            alpha=0.9,
        )

    # La campagne principale court des positions 0 a 5 (octobre a mars)
    ax.axvspan(-0.5, 5.5, color=VERT, alpha=0.07)
    # On agrandit d'abord l'axe, sinon le libelle chevauche la barre la plus haute
    ax.set_ylim(0, float(np.nanmax(pivot.to_numpy())) * 1.22)
    # Le libelle est place au-dessus des barres, dans l'espace libere ci-dessus
    ax.text(2.5, ax.get_ylim()[1] * 0.96, "Campagne principale",
            ha="center", va="top", fontsize=9, color=VERT, fontweight="bold")

    ax.set_title("Saisonnalite des apports")
    ax.set_ylabel("Tonnes")
    ax.set_xticks(positions)
    ax.set_xticklabels(etiquettes.values, rotation=45, ha="right", fontsize=8)
    ax.legend(fontsize=8)


# ---------------------------------------------------------------------------
# Graphique 3 : evolution des prix
# ---------------------------------------------------------------------------
def graphique_evolution_prix(ax) -> None:
    """
    Prix moyen pondere au fil de la campagne, une courbe par saison.

    Le prix affiche est SUM(montant) / SUM(tonnage), et non la moyenne des
    prix unitaires : une pesee de 12 kg ne doit pas peser autant qu'une pesee
    de 2 000 kg.
    """
    df = numeriser(
        charger_etape("resultat_saisonnalite"),
        ["prix_moyen_pondere", "part_grade_a_pct", "mois_campagne"],
    )

    etiquettes = (
        df.drop_duplicates("mois_campagne")
        .set_index("mois_campagne")["nom_mois"]
        .sort_index()
    )

    for saison, couleur in zip(sorted(df["saison"].unique()), [BRUN_FONCE, OCRE]):
        serie = df[df["saison"] == saison].sort_values("mois_campagne")
        ax.plot(
            serie["mois_campagne"], serie["prix_moyen_pondere"],
            marker="o", markersize=5, linewidth=2, color=couleur, label=saison,
        )

    ax.axvspan(0.5, 6.5, color=VERT, alpha=0.07)
    ax.set_title("Evolution du prix moyen pondere")
    ax.set_ylabel("FCFA par kg")
    ax.set_xticks(sorted(df["mois_campagne"].unique()))
    ax.set_xticklabels(etiquettes.values, rotation=45, ha="right", fontsize=8)
    ax.legend(fontsize=8)


# ---------------------------------------------------------------------------
# Graphique 4 : distribution des qualites
# ---------------------------------------------------------------------------
def graphique_qualites(ax) -> None:
    """
    Tonnage par grade, avec le taux de conformite a la norme d'exportation.

    Deux informations sur un seul graphique : combien pese chaque grade, et
    quelle part respecte le seuil de 8 % d'humidite.
    """
    df = numeriser(
        charger_etape("resultat_prix_qualite"),
        ["tonnes", "conforme_export_pct", "prix_moyen_pondere", "rang"],
    ).sort_values("rang")

    couleurs = [COULEURS_GRADE.get(g, OCRE) for g in df["nom_qualite"]]
    barres = ax.bar(df["nom_qualite"], df["tonnes"], color=couleurs, alpha=0.9)

    for barre, conforme, prix in zip(
        barres, df["conforme_export_pct"], df["prix_moyen_pondere"]
    ):
        ax.text(
            barre.get_x() + barre.get_width() / 2, barre.get_height() * 1.02,
            f"{conforme:.0f} % conformes\n{prix:.0f} FCFA/kg",
            ha="center", fontsize=8,
        )

    ax.set_title("Distribution des qualites et conformite export")
    ax.set_ylabel("Tonnes")
    ax.set_ylim(0, df["tonnes"].max() * 1.25)
    ax.tick_params(axis="x", labelsize=9)


# ---------------------------------------------------------------------------
# Graphique 5 : prime bio
# ---------------------------------------------------------------------------
def graphique_prime_bio(ax) -> None:
    """Prix moyen pondere selon la certification, a grade egal."""
    df = numeriser(charger_etape("resultat_prime_bio"), ["prix_moyen_pondere", "tonnes"])

    # certifie_bio peut arriver en booleen ou en texte selon la source
    df["certifie_bio"] = df["certifie_bio"].astype(str).str.lower().isin(["true", "1", "oui"])

    pivot = df.pivot_table(
        index="nom_qualite", columns="certifie_bio", values="prix_moyen_pondere"
    )
    ordre = ["Grade A", "Grade B", "Grade C", "Hors grade"]
    pivot = pivot.reindex([g for g in ordre if g in pivot.index])

    positions = np.arange(len(pivot))
    largeur = 0.38

    b1 = ax.bar(positions - largeur / 2, pivot[False], largeur,
                label="Conventionnel", color=OCRE, alpha=0.9)
    b2 = ax.bar(positions + largeur / 2, pivot[True], largeur,
                label="Certifie bio", color=VERT, alpha=0.9)

    # La prime en pourcentage est ce qui interesse le planteur
    for position, (sans, avec) in enumerate(zip(pivot[False], pivot[True])):
        prime = (avec / sans - 1) * 100
        ax.text(position, max(sans, avec) * 1.03, f"+{prime:.0f} %",
                ha="center", fontsize=9, fontweight="bold", color=VERT)

    annoter_barres(ax, list(b1) + list(b2), list(pivot[False]) + list(pivot[True]),
                   format_texte="{:.0f}", decalage=-0.14)

    ax.set_title("Effet de la certification bio, a grade egal")
    ax.set_ylabel("FCFA par kg")
    ax.set_xticks(positions)
    ax.set_xticklabels(pivot.index, fontsize=9)
    ax.set_ylim(0, pivot.max().max() * 1.2)
    ax.legend(fontsize=8)


# ---------------------------------------------------------------------------
# Graphique 6 : classement des cooperatives
# ---------------------------------------------------------------------------
def graphique_cooperatives(ax) -> None:
    """Top 10 des cooperatives par tonnage, la couleur indiquant la part bio."""
    df = numeriser(
        charger_etape("resultat_cooperatives"),
        ["tonnes", "part_bio_pct", "kg_par_planteur"],
    ).head(10).sort_values("tonnes")

    # Degrade du clair (peu de bio) au vert fonce (beaucoup de bio).
    # L'echelle est calee sur les valeurs observees, et non sur un maximum
    # arbitraire : sinon toutes les cooperatives peu certifiees recevraient
    # la meme teinte et le degrade n'apprendrait plus rien.
    parts = df["part_bio_pct"]
    etendue = max(parts.max() - parts.min(), 1e-9)
    palette = plt.cm.YlGn(0.25 + 0.7 * (parts - parts.min()) / etendue)
    barres = ax.barh(df["code_cooperative"], df["tonnes"], color=palette)

    annoter_barres(ax, barres, df["part_bio_pct"],
                   format_texte="{:.0f} % bio", horizontal=True)

    ax.set_title("Top 10 des cooperatives collectrices")
    ax.set_xlabel("Tonnes collectees")
    ax.set_xlim(0, df["tonnes"].max() * 1.2)
    ax.tick_params(axis="y", labelsize=8)


# ---------------------------------------------------------------------------
# Assemblage
# ---------------------------------------------------------------------------
GRAPHIQUES = [
    ("g1_production_region", "Production par region", graphique_production_region),
    ("g2_saisonnalite", "Saisonnalite des apports", graphique_saisonnalite),
    ("g3_evolution_prix", "Evolution des prix", graphique_evolution_prix),
    ("g4_qualites", "Distribution des qualites", graphique_qualites),
    ("g5_prime_bio", "Prime a la certification bio", graphique_prime_bio),
    ("g6_cooperatives", "Classement des cooperatives", graphique_cooperatives),
]

In [37]:
# ---------------------------------------------------------------------------
# Assemblage de la planche
# ---------------------------------------------------------------------------
titre("7. Tableau de bord")

figure = plt.figure(figsize=(16, 17))
figure.suptitle("Filiere cacao ivoirienne : tableau de bord des pesees",
                fontsize=17, fontweight="bold", y=0.995)

for position, (_, _, tracer) in enumerate(GRAPHIQUES, start=1):
    tracer(figure.add_subplot(3, 2, position))

figure.tight_layout(rect=(0, 0.01, 1, 0.985))

chemin_png = DATA_OUTPUT / "dashboard_cacao.png"
figure.savefig(chemin_png, dpi=DPI, bbox_inches="tight", facecolor=figure.get_facecolor())
ok(f"Tableau de bord enregistre : {chemin_png.name} ({taille_lisible(chemin_png)}, {DPI} dpi)")
plt.show()

╭────────────────────╮
│ 7. Tableau de bord │
╰────────────────────╯

OK Tableau de bord enregistre : dashboard_cacao.png (181.1 Ko, 150 dpi)

/var/folders/jt/7rnf8g3j7kq9frlv5xd583380000gn/T/ipykernel_5458/918721756.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Synthèse et rapport final

In [38]:
# ---------------------------------------------------------------------------
# KPI du pipeline, exportes en JSON
# ---------------------------------------------------------------------------
titre("8. Synthese du pipeline")

succes = pesees  # toutes les pesees sont valides apres nettoyage

rapport_final = {
    "meta": {
        "projet": "Pipeline ETL filiere cacao ivoirienne",
        "genere_le": datetime.now().isoformat(timespec="seconds"),
        "periode_debut": str(pesees["date"].min().date()),
        "periode_fin": str(pesees["date"].max().date()),
        "source_requetes": "supabase" if engine is not None else "duckdb_local",
    },
    "volumes": {
        "lignes_extraites": nb_depart,
        "lignes_conservees": len(pesees),
        "doublons_supprimes": stats_doublons["doublons_supprimes"],
        "prix_imputes": stats_imputation["prix_impute"],
        "humidites_imputees": stats_imputation["humidite_imputee"],
        "colonnes_finales": len(pesees.columns),
    },
    "kpi": {
        "tonnage_total_kg": int(pesees["tonnage_kg"].sum()),
        "valeur_totale_fcfa": int(pesees["montant_fcfa"].sum()),
        "prix_moyen_pondere": round(
            pesees["montant_fcfa"].sum() / pesees["tonnage_kg"].sum(), 1),
        "part_conforme_export_pct": round(pesees["conforme_export"].mean() * 100, 1),
        "nb_planteurs": int(pesees["id_planteur"].nunique()),
        "nb_cooperatives": int(pesees["cooperative"].nunique()),
    },
    "qualite": resultat.en_dict(),
    "anomalies": anomalies.to_dict(orient="records"),
}

chemin_rapport = DATA_OUTPUT / "rapport_final.json"
chemin_rapport.write_text(
    json.dumps(rapport_final, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

table = Table(box=None, show_header=False)
table.add_column(style="cyan")
table.add_column(justify="right")
for libelle, valeur in [
    ("Lignes extraites", f"{rapport_final['volumes']['lignes_extraites']:,}"),
    ("Lignes conservees", f"{rapport_final['volumes']['lignes_conservees']:,}"),
    ("Tonnage total", f"{rapport_final['kpi']['tonnage_total_kg'] / 1000:,.0f} tonnes"),
    ("Valeur totale", f"{rapport_final['kpi']['valeur_totale_fcfa'] / 1e9:,.2f} milliards FCFA"),
    ("Prix moyen pondere", f"{rapport_final['kpi']['prix_moyen_pondere']:,.0f} FCFA/kg"),
    ("Conformite export", f"{rapport_final['kpi']['part_conforme_export_pct']} %"),
    ("Planteurs actifs", f"{rapport_final['kpi']['nb_planteurs']:,}"),
    ("Regles qualite passees", f"{len(resultat.controles)}"),
]:
    table.add_row(libelle, valeur)

console.print(Panel(table, title="[bold]Pipeline cacao : synthese[/]", border_style="green"))

if engine is not None:
    engine.dispose()
    ok("Connexion Supabase fermee")

console.print("\n[bold green]Pipeline execute de bout en bout.[/]")

╭─────────────────────────╮
│ 8. Synthese du pipeline │
╰─────────────────────────╯

╭─────────────────────────────────────────── Pipeline cacao : synthese ───────────────────────────────────────────╮
│  Lignes extraites                      80,000                                                                   │
│  Lignes conservees                     78,800                                                                   │
│  Tonnage total                  24,345 tonnes                                                                   │
│  Valeur totale           20.73 milliards FCFA                                                                   │
│  Prix moyen pondere               851 FCFA/kg                                                                   │
│  Conformite export                     70.5 %                                                                   │
│  Planteurs actifs                       4,995                                                                   │
│  Regles qualite passees                     9                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Pipeline execute de bout en bout.

## 9. Conclusion

### Ce que le pipeline apporte

Le Conseil du Café-Cacao dispose désormais d'un entrepôt interrogeable en SQL, alimenté
automatiquement chaque nuit à 22h00 par un DAG Airflow, avec sept contrôles de qualité
bloquants avant tout chargement.

Trois constats métier en sortent directement : la concentration logistique sur le port de
San Pedro, la non-conformité massive du Grade C à la norme d'humidité, et la prime de
10 % attachée à la certification bio.

### Difficultés rencontrées

Le générateur fourni produisait des données incohérentes, ce qui a imposé un audit et une
reconstruction avant même de commencer le pipeline. Le tri chronologique d'une campagne
agricole, qui court d'octobre à septembre, a demandé une colonne dédiée dans la dimension
calendrier. Enfin, le conteneur Airflow impose ses propres versions de pandas et de
SQLAlchemy via un fichier de contraintes, ce qui a fait échouer le premier build.

### Limites

Les données sont synthétiques : les ordres de grandeur ont été calés sur la filière réelle
mais ne reflètent pas la production nationale. Les deux campagnes se ressemblent beaucoup,
là où des aléas climatiques créeraient des écarts. L'imputation de l'humidité par la
médiane du grade est circulaire, puisque c'est l'humidité qui détermine le grade.

### Améliorations possibles

Historiser les dimensions pour suivre un planteur qui change de coopérative, ajouter une
dimension agent de bascule pour suivre la qualité de la saisie, et brancher le pipeline
sur les fichiers réels des coopératives à la place du générateur.